In [2]:

#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
paper_4.2.4.4_all_build_integrated.py
-------------------------------------
4.2.4.4節（OH系とFP系のクラスタ対応関係）の図表を“一括で”再現・比較・検証する統合スクリプト。

含まれる機能（元コードとの対応）:
  (A/F/D) OH→FP 主分布・ペア一貫性・OHクラスタ一貫性・図表（本文/付録仕分け）・サマリー
  (B/E)   FP→OH（逆方向）主分布・ペア一貫性・FPクラスタ一貫性・図表（本文/付録仕分け）・サマリー
  (E)     signless と new の一致度（ARI/NMI/AMI）: OH/FP それぞれで集計し本文/付録図表を出力
  (E)     双方向サマリー（OH→FPのスコア vs FP→OHの複合スコア）の表・図
  (C)     k固定検証（Permutation 検定・Bootstrap CI・k=2限定/その他での集約と図）※任意実行

出力ルート（分かりやすさを重視した構成）:
  ROOT/paper_4.2.4.4_OHFP/
    ├─ main_text/                 ← 本文用（cumeig×gap / cumeig×db を中心）
    ├─ appendix/                  ← 付録（その他条件・全Kなど）
    ├─ analysis_csv/              ← OH→FP 前方向のCSV
    ├─ reverse/
    │    ├─ main_text/            ← 逆方向の本文図
    │    ├─ appendix/             ← 逆方向の付録図
    │    └─ analysis_csv/         ← 逆方向のCSV
    ├─ bidirectional_summary/     ← 双方向まとめ
    └─ validation/                ← k固定・Permutation・Bootstrap（任意）

※ 実行前に ROOT を環境に合わせて調整してください。
"""

from pathlib import Path
import os, re
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
from matplotlib.patches import Patch
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, adjusted_mutual_info_score
from itertools import combinations
from scipy.spatial.distance import cosine

# ========= ユーザー設定 =========
ROOT = Path("/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No")
FIGS_OH = ROOT / "figs_OH"
FIGS_FP = ROOT / "figs_FP"

# signless 側（必要に応じて調整）
SIGNLESS_BASE = ROOT / "STEP3.2_signlessCorr_MDS_20250904"
SIGNLESS_DIR = {"OH": SIGNLESS_BASE / "OH", "FP": SIGNLESS_BASE / "FP"}

# features（材料列のみ）
FEATURES_OH = ROOT / "features_OH_onlyMat.csv"
FEATURES_FP = ROOT / "features_FP_onlyMat.csv"

# 出力ルート
OUT_OHFP_ROOT   = ROOT / "paper_4.2.4.4_OHFP_20251016_comp"
OUT_MAIN        = OUT_OHFP_ROOT / "main_text"
OUT_APPENDIX    = OUT_OHFP_ROOT / "appendix"
OUT_VARMAP      = OUT_OHFP_ROOT / "analysis_csv"
OUT_R_ROOT      = OUT_OHFP_ROOT / "reverse"
OUT_R_MAIN      = OUT_R_ROOT / "main_text"
OUT_R_APPENDIX  = OUT_R_ROOT / "appendix"
OUT_R_CSV       = OUT_R_ROOT / "analysis_csv"
OUT_BIDIR       = OUT_OHFP_ROOT / "bidirectional_summary"
OUT_VALID       = OUT_OHFP_ROOT / "validation"
for d in [OUT_MAIN, OUT_APPENDIX, OUT_VARMAP, OUT_R_MAIN, OUT_R_APPENDIX, OUT_R_CSV, OUT_BIDIR, OUT_VALID]:
    d.mkdir(parents=True, exist_ok=True)

# 対象の mode/index
DIMS    = ["top3","cumeig"]
INDICES = ["silhouette","dunn","gap","ch","db","ptbiserial"]
MAIN_PAIRS = {("cumeig","gap"), ("cumeig","db")}  # 本文優先

# しきい値／定数
FRAG_PREV_THRESHOLD = 0.05   # P(fragment=1 | material=1) 下限（前方向）
MAT_PREV_THRESHOLD  = 0.05   # P(material=1 | fragment=1) 下限（逆方向）
PURITY_FLAG_THR     = 0.60
COS_SIM_THR         = 0.60
JS_DIV_THR          = 0.50
K_MAX_KEEP          = 30
DPI                 = 300

# 検証（任意実行）の反復数
N_PERM = 500
N_BOOT = 500
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

# ========= フォント =========
def set_font():
    cand = ["Noto Sans CJK JP","Noto Sans JP","Source Han Sans JP","IPAexGothic","Yu Gothic","Meiryo","Hiragino Sans","Meiryo UI"]
    have = set()
    for p in fm.findSystemFonts(fontext="ttf"):
        try:
            have.add(fm.FontProperties(fname=p).get_name())
        except:
            pass
    for w in cand:
        if any(w.lower() in h.lower() for h in have):
            matplotlib.rcParams["font.family"] = w
            break
    matplotlib.rcParams["axes.unicode_minus"] = False
    matplotlib.rcParams.update({"font.size":10, "axes.titlesize":12, "axes.labelsize":10, "legend.fontsize":9})

# ========= IO =========
def read_csv_guess(path: Path) -> pd.DataFrame:
    if not path or not path.exists():
        raise FileNotFoundError(path)
    for enc in ("utf-8","utf-8-sig","cp932"):
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            continue
    raise RuntimeError(f"Failed to read: {path}")

def read_csv_quiet(path: Path) -> pd.DataFrame | None:
    if not path or not path.exists():
        return None
    for enc in ("utf-8","utf-8-sig","cp932"):
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            continue
    return None

def load_features_bin(path: Path) -> pd.DataFrame:
    df = read_csv_guess(path)
    if df.columns[0].lower() != "sample_id":
        df = df.rename(columns={df.columns[0]:"sample_id"})
    X = df.set_index("sample_id").apply(pd.to_numeric, errors="coerce").fillna(0.0)
    return (X != 0).astype(int)

# ========= 既存 ClusterAssign の最新検出 =========
FN_RE = re.compile(
    r"^ClusterAssign_(?P<mode>top3|cumeig)_(?P<index>silhouette|dunn|gap|ch|db|ptbiserial)_(?P<ds>OH|FP)_(?P<date>\d{8})_(?P<hm>\d{4})\.csv$"
)

def latest_by_mode_index(fig_dir: Path, ds: str):
    latest = {}
    for p in fig_dir.glob("ClusterAssign_*.csv"):
        m = FN_RE.match(p.name)
        if not m or m["ds"] != ds:
            continue
        key = (m["mode"], m["index"])
        ts  = f"{m['date']}_{m['hm']}"
        if key not in latest or ts > latest[key][0]:
            latest[key] = (ts, p)
    return {k: v[1] for k,v in latest.items()}

def read_var_cluster(path: Path) -> pd.Series:
    df = read_csv_guess(path)
    cols = {c.lower(): c for c in df.columns}
    vcol = cols.get("variable", df.columns[0])
    ccol = cols.get("cluster",  df.columns[-1])
    s = pd.Series(df[ccol].values, index=df[vcol].astype(str).values, name="cluster")
    try:
        s = s.astype(int)
    except:
        pass
    return s

# ========= 指標関数 =========
def entropy_from_dist(dist: dict[int, float]) -> float:
    if not dist:
        return np.nan
    p = np.array(list(dist.values()), dtype=float)
    p = p / (p.sum() + 1e-12)
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())

def js_divergence(p, q):
    p = np.asarray(p, dtype=float); p = p/p.sum() if p.sum()>0 else p
    q = np.asarray(q, dtype=float); q = q/q.sum() if q.sum()>0 else q
    m = 0.5*(p+q)
    def _kl(a,b):
        a = np.clip(a,1e-12,1); b = np.clip(b,1e-12,1)
        return float(np.sum(a*(np.log2(a)-np.log2(b))))
    return 0.5*_kl(p,m) + 0.5*_kl(q,m)

def material_fp_purity_entropy_from_samples(material, Xoh, lab_fp):
    """
    material: 材料列名
    Xoh: (samples × materials) 0/1
    lab_fp: pd.Series(index=samples, values=FP_cluster int)  # derive_sample_labels() から得る
    """
    idx = Xoh.index[Xoh[material] == 1]
    s = lab_fp.loc[idx].dropna().astype(int)
    n = int(s.shape[0])
    if n == 0:
        return np.nan, np.nan, 0, {}
    counts = s.value_counts().sort_index()
    p = (counts / counts.sum()).to_dict()                # FPクラスタ分布（サンプル基準）
    purity = float(max(p.values()))
    H = float(-sum(pi * np.log2(pi) for pi in p.values() if pi > 0))  # log2
    return purity, H, n, p


# ========= サンプル優勢クラスタ（投票） =========
def derive_sample_labels(features: pd.DataFrame, var2clu: pd.Series, min_votes=1) -> pd.Series:
    common = [v for v in features.columns if v in var2clu.index]
    if not common:
        return pd.Series(index=features.index, dtype="float64")
    clus = pd.Series(var2clu.loc[common]).astype(str)
    uniq = sorted(clus.unique())
    mat = features[common].values
    key2idx = {k:i for i,k in enumerate(uniq)}
    col2cidx = np.array([key2idx[c] for c in clus.values], dtype=int)
    votes = np.zeros((features.shape[0], len(uniq)), dtype=int)
    rows, cols = np.where(mat==1)
    for r, c in zip(rows, cols):
        votes[r, col2cidx[c]] += 1
    maxv = votes.max(axis=1)
    labels = np.where(maxv>=min_votes, votes.argmax(axis=1), -1)
    out = pd.Series(labels, index=features.index).replace({-1: np.nan})
    lbl_vals = [uniq[int(i)] if pd.notna(i) else np.nan for i in out.values]
    try:
        return pd.Series(lbl_vals, index=features.index).astype(int)
    except:
        return pd.Series(lbl_vals, index=features.index)

# ========= 前方向（OH→FP）: material → FP 分布 =========
def material_to_fp_distribution(material, Xoh, Xfp, fp_var2clu, threshold=FRAG_PREV_THRESHOLD):
    idx = Xoh.index[Xoh[material] == 1]
    if len(idx) == 0:
        return None, None, 0, {}
    prev = Xfp.loc[idx].mean(axis=0)  # P(fragment=1 | material=1)
    sel = prev[prev >= threshold]
    if sel.empty:
        return None, None, len(idx), {}
    df = pd.DataFrame({"frag": sel.index, "w": sel.values})
    df["fp_cluster"] = df["frag"].map(fp_var2clu).astype("Int64")
    df = df.dropna(subset=["fp_cluster"])
    if df.empty:
        return None, None, len(idx), {}
    grp = df.groupby("fp_cluster", as_index=False)["w"].sum()
    grp["p"] = grp["w"]/grp["w"].sum()
    dist = dict(zip(grp["fp_cluster"].astype(int), grp["p"]))
    maj = int(grp.sort_values("p", ascending=False).iloc[0]["fp_cluster"])
    purity = float(grp["p"].max())
    return maj, purity, len(idx), dist

# ========= 逆方向（FP→OH）: fragment → OH 分布 =========
def fragment_to_oh_distribution(fragment, Xfp, Xoh, oh_var2clu, threshold=MAT_PREV_THRESHOLD):
    idx = Xfp.index[Xfp[fragment] == 1]
    if len(idx) == 0:
        return None, None, 0, {}
    prev = Xoh.loc[idx].mean(axis=0)  # P(material=1 | fragment=1)
    sel = prev[prev >= threshold]
    if sel.empty:
        return None, None, len(idx), {}
    df = pd.DataFrame({"mat": sel.index, "w": sel.values})
    df["oh_cluster"] = df["mat"].map(oh_var2clu).astype("Int64")
    df = df.dropna(subset=["oh_cluster"])
    if df.empty:
        return None, None, len(idx), {}
    grp = df.groupby("oh_cluster", as_index=False)["w"].sum()
    grp["p"] = grp["w"]/grp["w"].sum()
    dist = dict(zip(grp["oh_cluster"].astype(int), grp["p"]))
    maj = int(grp.sort_values("p", ascending=False).iloc[0]["oh_cluster"])
    purity = float(grp["p"].max())
    return maj, purity, len(idx), dist

# ========= 図（前方向/逆方向で再利用） =========
def _plot_pair_hist(ax, series: pd.Series, title: str, xlabel: str):
    s = series.dropna()
    if len(s):
        ax.hist(s, bins=20, alpha=0.9)
        ax.set_title(title); ax.set_xlabel(xlabel); ax.set_ylabel("Count")

def plot_material_scatter(df_mat: pd.DataFrame, outdir: Path, mode: str, index: str, top_labels:int=15, main_text:bool=False):
    set_font()
    if df_mat is None or df_mat.empty: return
    size = 30 + 3 * df_mat["n_samples_with_material"].fillna(0).astype(float)
    c    = df_mat["OH_cluster"].astype("category")
    fig, ax = plt.subplots(figsize=(9,6), dpi=DPI)
    ax.scatter(df_mat["FP_entropy"], df_mat["FP_purity"], c=c.cat.codes, s=size, cmap="tab20", alpha=0.85, edgecolors="none")
    ax.set_xlabel("FP entropy (larger = more fragmented)")
    ax.set_ylabel("FP purity (larger = more cohesive)")
    title = f"Fig 4.2.4.4: Materials — Cohesion vs Fragmentation ({mode} × {index})"
    if main_text: title += " [Main]"
    ax.set_title(title)
    lab_df = df_mat.sort_values(["FP_purity","n_samples_with_material"], ascending=False).head(top_labels)
    for _, r in lab_df.iterrows():
        ax.text(r["FP_entropy"], r["FP_purity"]+0.015, str(r["material"]), ha="center", va="bottom", fontsize=8)
    ax.grid(True, linestyle="--", linewidth=0.4, alpha=0.6)
    outdir.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fname = f"Fig_4.2.4.4_materials_entropy-vs-purity_{mode}_{index}{'_MAIN' if main_text else ''}"
    fig.savefig(outdir / f"{fname}.png", bbox_inches="tight"); fig.savefig(outdir / f"{fname}.pdf", bbox_inches="tight")
    plt.close(fig)

def plot_fragment_scatter(df_frag: pd.DataFrame, outdir: Path, mode: str, index: str, top_labels:int=15, main_text:bool=False):
    set_font()
    if df_frag is None or df_frag.empty:
        return
    size = 30 + 3 * df_frag["n_samples_with_fragment"].fillna(0).astype(float)
    c    = df_frag["FP_cluster"].astype("category")
    fig, ax = plt.subplots(figsize=(9,6), dpi=DPI)
    ax.scatter(df_frag["OH_entropy"], df_frag["OH_purity"],
               c=c.cat.codes, s=size, cmap="tab20", alpha=0.85, edgecolors="none")
    ax.set_xlabel("OH entropy (larger = more fragmented)")
    ax.set_ylabel("OH purity (larger = more cohesive)")
    title = f"Fig 4.2.4.4R: Fragments — Cohesion vs Fragmentation ({mode} × {index})"
    if main_text: title += " [Main]"
    ax.set_title(title)
    lab_df = df_frag.sort_values(["OH_purity","n_samples_with_fragment"], ascending=False).head(top_labels)
    for _, r in lab_df.iterrows():
        ax.text(r["OH_entropy"], r["OH_purity"]+0.015, str(r["fragment"]),
                ha="center", va="bottom", fontsize=8)
    ax.grid(True, linestyle="--", linewidth=0.4, alpha=0.6)
    outdir.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fname = f"Fig_4.2.4.4R_fragments_entropy-vs-purity_{mode}_{index}{'_MAIN' if main_text else ''}"
    fig.savefig(outdir / f"{fname}.png", bbox_inches="tight"); fig.savefig(outdir / f"{fname}.pdf", bbox_inches="tight")
    plt.close(fig)

def plot_fragment_scatter(df_frag: pd.DataFrame, outdir: Path, mode: str, index: str, top_labels:int=15, main_text:bool=False):
    set_font()
    if df_frag is None or df_frag.empty: 
        return
    size = 30 + 3 * df_frag["n_samples_with_fragment"].fillna(0).astype(float)
    c    = df_frag["FP_cluster"].astype("category")
    fig, ax = plt.subplots(figsize=(9,6), dpi=DPI)
    sc = ax.scatter(
        df_frag["OH_entropy"], 
        df_frag["OH_purity"], 
        c=c.cat.codes, s=size, cmap="tab20", alpha=0.85, edgecolors="none"
    )
    ax.set_xlabel("OH entropy (larger = more materials-spread)")
    ax.set_ylabel("OH purity (larger = more materials-specific)")
    title = f"Fig 4.2.4.4R: Fragments — Specificity vs Spread ({mode} × {index})"
    if main_text: 
        title += " [Main]"
    ax.set_title(title)

    # ラベルは上位だけ
    lab_df = df_frag.sort_values(
        ["OH_purity","n_samples_with_fragment"], ascending=False
    ).head(top_labels)
    for _, r in lab_df.iterrows():
        ax.text(
            r["OH_entropy"], r["OH_purity"]+0.015, 
            str(r["fragment"]), ha="center", va="bottom", fontsize=8
        )

    ax.grid(True, linestyle="--", linewidth=0.4, alpha=0.6)
    outdir.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fname = f"Fig_4.2.4.4R_fragments_specificity-vs-spread_{mode}_{index}{'_MAIN' if main_text else ''}"
    fig.savefig(outdir / f"{fname}.png", bbox_inches="tight")
    fig.savefig(outdir / f"{fname}.pdf", bbox_inches="tight")
    plt.close(fig)

def plot_pair_distributions_forward(df_pair: pd.DataFrame, outdir: Path, mode: str, index: str, main_text:bool=False):
    if df_pair is None or df_pair.empty: return
    set_font()
    fig, ax = plt.subplots(figsize=(8,4), dpi=DPI)
    rate = df_pair["FP_major_same"].value_counts(normalize=True).rename({True:"Same", False:"Different"})
    idx = ["Same","Different"]; vals = [rate.get("Same",0.0), rate.get("Different",0.0)]
    ax.bar(idx, vals, color=["#4E79A7","#E15759"], alpha=0.9)
    ax.set_ylim(0,1); ax.set_ylabel("Proportion")
    title = f"Fig 4.2.4.4: Pair-level — FP major same? ({mode} × {index})"
    if main_text: title += " [Main]"
    ax.set_title(title)
    for xi, val in enumerate(vals): ax.text(xi, val+0.02, f"{val:.2f}", ha="center", va="bottom")
    fig.tight_layout()
    fname = f"Fig_4.2.4.4_pair_major-same_{mode}_{index}{'_MAIN' if main_text else ''}"
    fig.savefig(outdir / f"{fname}.png", bbox_inches="tight"); fig.savefig(outdir / f"{fname}.pdf", bbox_inches="tight")
    plt.close(fig)

    for col, xlabel in [("FP_cosine_similarity","Cosine similarity (1=similar)"),
                        ("FP_JS_divergence","JS divergence (0=similar)")]:
        s = df_pair[col].dropna()
        if not len(s): continue
        fig, ax = plt.subplots(figsize=(8,4), dpi=DPI)
        _plot_pair_hist(ax, s, f"Fig 4.2.4.4: Pair-level — {col} ({mode} × {index})" + (" [Main]" if main_text else ""), xlabel)
        fig.tight_layout()
        fname = f"Fig_4.2.4.4_pair_{col}_hist_{mode}_{index}{'_MAIN' if main_text else ''}"
        fig.savefig(outdir / f"{fname}.png", bbox_inches="tight"); fig.savefig(outdir / f"{fname}.pdf", bbox_inches="tight")
        plt.close(fig)

def plot_pair_distributions_reverse(df_pair: pd.DataFrame, outdir: Path, mode: str, index: str, main_text:bool=False):
    if df_pair is None or df_pair.empty: return
    set_font()
    fig, ax = plt.subplots(figsize=(8,4), dpi=DPI)
    rate = df_pair["OH_major_same"].value_counts(normalize=True).rename({True:"Same", False:"Different"})
    idx = ["Same","Different"]; vals = [rate.get("Same",0.0), rate.get("Different",0.0)]
    ax.bar(idx, vals, color=["#4E79A7","#E15759"], alpha=0.9)
    ax.set_ylim(0,1); ax.set_ylabel("Proportion")
    title = f"Fig 4.2.4.4R: Pair-level — OH major same? ({mode} × {index})"
    if main_text: title += " [Main]"
    ax.set_title(title)
    for xi, val in enumerate(vals): ax.text(xi, val+0.02, f"{val:.2f}", ha="center", va="bottom")
    fig.tight_layout()
    fname = f"Fig_4.2.4.4R_pair_major-same_{mode}_{index}{'_MAIN' if main_text else ''}"
    fig.savefig(outdir / f"{fname}.png", bbox_inches="tight"); fig.savefig(outdir / f"{fname}.pdf", bbox_inches="tight")
    plt.close(fig)

    for col, xlabel in [("OH_cosine_similarity","Cosine similarity (1=similar)"),
                        ("OH_JS_divergence","JS divergence (0=similar)")]:
        s = df_pair[col].dropna()
        if not len(s): continue
        fig, ax = plt.subplots(figsize=(8,4), dpi=DPI)
        _plot_pair_hist(ax, s, f"Fig 4.2.4.4R: Pair-level — {col} ({mode} × {index})" + (" [Main]" if main_text else ""), xlabel)
        fig.tight_layout()
        fname = f"Fig_4.2.4.4R_pair_{col}_hist_{mode}_{index}{'_MAIN' if main_text else ''}"
        fig.savefig(outdir / f"{fname}.png", bbox_inches="tight"); fig.savefig(outdir / f"{fname}.pdf", bbox_inches="tight")
        plt.close(fig)

def plot_ohcluster_consistency(df_ohc: pd.DataFrame, outdir: Path, mode: str, index: str, main_text:bool=False):
    if df_ohc is None or df_ohc.empty: return
    set_font()
    df = df_ohc.sort_values("cohesive_ratio", ascending=False).copy()
    fig, ax = plt.subplots(figsize=(8,4), dpi=DPI)
    x = np.arange(len(df))
    ax.bar(x, df["cohesive_ratio"].values, color="#59A14F", alpha=0.9)
    ax.set_xticks(x); ax.set_xticklabels([f"OH {int(c)}" for c in df["OH_cluster"]], rotation=35, ha="right")
    ax.set_ylim(0,1.05); ax.set_ylabel("Cohesive ratio (pair-level)")
    title = f"Fig 4.2.4.4: OH-cluster cohesion in FP space ({mode} × {index})"
    if main_text: title += " [Main]"
    ax.set_title(title)
    for xi, v in zip(x, df["cohesive_ratio"].values): ax.text(xi, v+0.02, f"{v:.2f}", ha="center", va="bottom", fontsize=8)
    ax.grid(axis="y", linestyle="--", linewidth=0.4, alpha=0.6)
    fig.tight_layout()
    fname = f"Fig_4.2.4.4_OHcluster_cohesion_{mode}_{index}{'_MAIN' if main_text else ''}"
    fig.savefig(outdir / f"{fname}.png", bbox_inches="tight"); fig.savefig(outdir / f"{fname}.pdf", bbox_inches="tight")
    plt.close(fig)

def plot_fpcluster_consistency(df_fpc: pd.DataFrame, outdir: Path, mode: str, index: str, main_text:bool=False):
    if df_fpc is None or df_fpc.empty: return
    set_font()
    df = df_fpc.sort_values("cohesive_ratio", ascending=False).copy()
    fig, ax = plt.subplots(figsize=(8,4), dpi=DPI)
    x = np.arange(len(df))
    ax.bar(x, df["cohesive_ratio"].values, color="#59A14F", alpha=0.9)
    ax.set_xticks(x); ax.set_xticklabels([f"FP {int(c)}" for c in df["FP_cluster"]], rotation=35, ha="right")
    ax.set_ylim(0,1.05); ax.set_ylabel("Cohesive ratio (pair-level in OH space)")
    title = f"Fig 4.2.4.4R: FP-cluster cohesion in OH space ({mode} × {index})"
    if main_text: title += " [Main]"
    ax.set_title(title)
    for xi, v in zip(x, df["cohesive_ratio"].values): ax.text(xi, v+0.02, f"{v:.2f}", ha="center", va="bottom", fontsize=8)
    ax.grid(axis="y", linestyle="--", linewidth=0.4, alpha=0.6)
    fig.tight_layout()
    fname = f"Fig_4.2.4.4R_FPcluster_cohesion_{mode}_{index}{'_MAIN' if main_text else ''}"
    fig.savefig(outdir / f"{fname}.png", bbox_inches="tight"); fig.savefig(outdir / f"{fname}.pdf", bbox_inches="tight")
    plt.close(fig)

def plot_mini_sankey_material_to_fp(df_mat: pd.DataFrame, outdir: Path, mode: str, index: str, topN=20, main_text:bool=False):
    if df_mat is None or df_mat.empty: return
    set_font()
    df = df_mat.sort_values("FP_purity", ascending=False).head(topN).copy()
    if df.empty: return
    widths = (df["FP_purity"] + 0.2) / (df["FP_purity"].max() + 0.2)
    y = 0.0
    fig, ax = plt.subplots(figsize=(10, 0.35*len(df)+2), dpi=DPI)
    for i, r in df.iterrows():
        w = widths.loc[i]
        ax.plot([0, 1], [y, y], lw=8*w, alpha=0.9, color="#4E79A7")
        ax.text(-0.02, y, str(r["material"]), ha="right", va="center", fontsize=8)
        fp_major = r.get("FP_major_cluster"); right_lbl = f"FP {int(fp_major)}" if pd.notna(fp_major) else "FP –"
        ax.text(1.02, y, right_lbl, ha="left", va="center", fontsize=8, color="#555")
        y += 0.5
    ax.set_xlim(-0.25, 1.25); ax.set_ylim(-0.5, y-0.0); ax.axis("off")
    title = f"Fig 4.2.4.4: Material → FP major cluster (top {topN}) ({mode} × {index})"
    if main_text: title += " [Main]"
    ax.set_title(title, loc="left")
    fig.tight_layout()
    fname = f"Fig_4.2.4.4_material-to-FPmajor_miniAlluvial_{mode}_{index}{'_MAIN' if main_text else ''}"
    fig.savefig(outdir / f"{fname}.png", bbox_inches="tight"); fig.savefig(outdir / f"{fname}.pdf", bbox_inches="tight")
    plt.close(fig)

def plot_mini_sankey_fragment_to_oh(df_frag: pd.DataFrame, outdir: Path, mode: str, index: str, topN=20, main_text:bool=False):
    if df_frag is None or df_frag.empty: return
    set_font()
    df = df_frag.sort_values("OH_purity", ascending=False).head(topN).copy()
    if df.empty: return
    widths = (df["OH_purity"] + 0.2) / (df["OH_purity"].max() + 0.2)
    y = 0.0
    fig, ax = plt.subplots(figsize=(10, 0.35*len(df)+2), dpi=DPI)
    for i, r in df.iterrows():
        w = widths.loc[i]
        ax.plot([0, 1], [y, y], lw=8*w, alpha=0.9, color="#E15759")
        ax.text(-0.02, y, str(r["fragment"]), ha="right", va="center", fontsize=8)
        oh_major = r.get("OH_major_cluster"); right_lbl = f"OH {int(oh_major)}" if pd.notna(oh_major) else "OH –"
        ax.text(1.02, y, right_lbl, ha="left", va="center", fontsize=8, color="#555")
        y += 0.5
    ax.set_xlim(-0.25, 1.25); ax.set_ylim(-0.5, y-0.0); ax.axis("off")
    title = f"Fig 4.2.4.4R: Fragment → OH major cluster (top {topN}) ({mode} × {index})"
    if main_text: title += " [Main]"
    ax.set_title(title, loc="left")
    fig.tight_layout()
    fname = f"Fig_4.2.4.4R_fragment-to-OHmajor_miniAlluvial_{mode}_{index}{'_MAIN' if main_text else ''}"
    fig.savefig(outdir / f"{fname}.png", bbox_inches="tight"); fig.savefig(outdir / f"{fname}.pdf", bbox_inches="tight")
    plt.close(fig)

# ========= OH→FP（前方向） =========
def build_forward_direction():
    set_font()
    Xoh0 = load_features_bin(FEATURES_OH)
    Xfp0 = load_features_bin(FEATURES_FP)
    common_samples = Xoh0.index.intersection(Xfp0.index)
    if len(common_samples)==0:
        raise RuntimeError("No common samples between features_OH_onlyMat and features_FP_onlyMat.")
    Xoh0 = Xoh0.loc[common_samples]; Xfp0 = Xfp0.loc[common_samples]

    oh_latest = latest_by_mode_index(FIGS_OH, "OH")
    fp_latest = latest_by_mode_index(FIGS_FP, "FP")
    keys = [(m,i) for m in DIMS for i in INDICES if (m,i) in oh_latest and (m,i) in fp_latest]
    if not keys:
        raise RuntimeError("OH/FP の ClusterAssign に共通 (mode,index) が見つかりません。")

    summary_rows = []

    for (mode, index) in keys:
        print(f"[FWD] {mode} × {index}")
        oh_assign = read_var_cluster(oh_latest[(mode,index)])
        fp_assign = read_var_cluster(fp_latest[(mode,index)])
        
        Xoh = Xoh0[[c for c in Xoh0.columns if c in oh_assign.index]]
        Xfp = Xfp0[[c for c in Xfp0.columns if c in fp_assign.index]]
        lab_fp = derive_sample_labels(Xfp, fp_assign, min_votes=1)


        # 材料 → FP 主分布
        materials = [m for m in oh_assign.index if m in Xoh0.columns]
        rows_mat = []
        for mat in materials:
            n_mat = int(Xoh[mat].sum())
            if n_mat <= 0:
                continue
            purity, H, n_s, dist = material_fp_purity_entropy_from_samples(mat, Xoh, lab_fp)
            oh_c = oh_assign.get(mat, np.nan)
            rows_mat.append({
                "mode": mode, "index": index,
                "material": mat,
                "OH_cluster": int(oh_c) if pd.notna(oh_c) else np.nan,
                "n_samples_with_material": n_s,
                "FP_major_cluster": (max(dist, key=dist.get) if dist else np.nan),
                "FP_purity": purity,
                "FP_entropy": H,
                "FP_k_effective": (len(dist) if dist else 0),
                "FP_dist": ";".join(f"{k}:{v:.3f}" for k,v in sorted(dist.items())) if dist else ""
            })

        df_mat = pd.DataFrame(rows_mat)
        out_csv1 = OUT_VARMAP / f"Table_4.2.4.4_material-to-FPmajor_{mode}_{index}.csv"
        if len(df_mat): df_mat.to_csv(out_csv1, index=False, encoding="utf-8-sig")

        # 材料ペア（同一OHクラスタ）で一貫性
        rows_pair = []
        if len(df_mat):
            def parse_dist(s):
                if isinstance(s, str) and s:
                    d={}; 
                    for t in s.split(";"):
                        k,v = t.split(":"); d[int(k)] = float(v)
                    return d
                return {}
            dists = {r.material: parse_dist(r.FP_dist) for r in df_mat.itertuples()}
            all_fp_clusters = sorted({k for d in dists.values() for k in d.keys()})
            def vec_of(dist):
                return np.array([dist.get(k, 0.0) for k in all_fp_clusters], dtype=float)

            for k, grp in df_mat.dropna(subset=["OH_cluster"]).groupby("OH_cluster"):
                mats = grp["material"].tolist()
                for a, b in combinations(mats, 2):
                    da, db = dists.get(a, {}), dists.get(b, {})
                    va, vb = vec_of(da), vec_of(db)
                    if va.sum()==0 and vb.sum()==0:
                        cos_sim = np.nan; jsd = np.nan
                    else:
                        cos_sim = 1.0 - (cosine(va, vb) if (va.sum()>0 and vb.sum()>0) else 1.0)
                        jsd = js_divergence(va, vb) if (va.sum()>0 and vb.sum()>0) else np.nan
                    maj_same = (df_mat.loc[df_mat["material"]==a, "FP_major_cluster"].values[0] ==
                                df_mat.loc[df_mat["material"]==b, "FP_major_cluster"].values[0])
                    purity_min = float(min(
                        df_mat.loc[df_mat["material"]==a, "FP_purity"].values[0] if len(df_mat.loc[df_mat["material"]==a, "FP_purity"]) else 0.0,
                        df_mat.loc[df_mat["material"]==b, "FP_purity"].values[0] if len(df_mat.loc[df_mat["material"]==b, "FP_purity"]) else 0.0
                    ))
                    rows_pair.append({
                        "mode": mode, "index": index,
                        "OH_cluster": int(k),
                        "material_A": a, "material_B": b,
                        "FP_major_same": bool(maj_same),
                        "FP_purity_min": purity_min,
                        "FP_cosine_similarity": cos_sim,
                        "FP_JS_divergence": jsd
                    })
        df_pair = pd.DataFrame(rows_pair)
        out_csv2 = OUT_VARMAP / f"Table_4.2.4.4_material-pair_cohesion_{mode}_{index}.csv"
        if len(df_pair): df_pair.to_csv(out_csv2, index=False, encoding="utf-8-sig")

        # OHクラスタ一貫性
        # --- PURE: FP_major 同一のみで計算（本文用） ---
        if len(df_pair):
            df_ohc_pure = (
                df_pair.groupby(["mode","index","OH_cluster"], as_index=False)
                       .agg(n_pairs=("FP_major_same","size"),
                            n_same=("FP_major_same","sum"))
            )
            df_ohc_pure["cohesive_ratio"] = df_ohc_pure["n_same"] / df_ohc_pure["n_pairs"].replace(0, np.nan)
        else:
            df_ohc_pure = pd.DataFrame()

        out_csv3_pure = OUT_VARMAP / f"Table_4.2.4.4_OHcluster_cohesion_PURE_{mode}_{index}.csv"
        if len(df_ohc_pure): df_ohc_pure.to_csv(out_csv3_pure, index=False, encoding="utf-8-sig")


        # 図（本文/付録の振分）
        is_main = (mode, index) in MAIN_PAIRS
        out_dir = OUT_MAIN / f"FigSet_4.2.4.4_{mode}_{index}" if is_main else OUT_APPENDIX / f"FigSet_4.2.4.4_{mode}_{index}"
        out_dir.mkdir(parents=True, exist_ok=True)
        plot_material_scatter(df_mat, out_dir, mode, index, top_labels=15, main_text=is_main)
        plot_pair_distributions_forward(df_pair, out_dir, mode, index, main_text=is_main)
        if is_main:
            plot_ohcluster_consistency(df_ohc_pure, out_dir, mode, index, main_text=True)
        else:    
            plot_ohcluster_consistency(df_ohc, out_dir, mode, index, main_text=is_main)
        plot_mini_sankey_material_to_fp(df_mat, out_dir, mode, index, topN=20, main_text=is_main)

        # サマリー
        summary_rows.append({
            "mode": mode, "index": index,
            "n_materials": len(df_mat),
            "mean_FP_purity": df_mat["FP_purity"].dropna().mean() if "FP_purity" in df_mat else np.nan,
            "mean_FP_entropy": df_mat["FP_entropy"].dropna().mean() if "FP_entropy" in df_mat else np.nan,
            "pair_FP_major_same_rate": df_pair["FP_major_same"].mean() if len(df_pair) else np.nan,
            "pair_mean_cosine_similarity": df_pair["FP_cosine_similarity"].dropna().mean() if len(df_pair) else np.nan,
            "pair_mean_JS_divergence": df_pair["FP_JS_divergence"].dropna().mean() if len(df_pair) else np.nan,
            # ★ ここを PURE に
            "OHcluster_median_cohesive_ratio": df_ohc_pure["cohesive_ratio"].median() if len(df_ohc_pure) else np.nan
        })


    # 全条件サマリー（前方向）
    if summary_rows:
        df_sum = pd.DataFrame(summary_rows).sort_values(["mode","index"])
        outsum_all = OUT_OHFP_ROOT / "Table_4.2.4.4_summary_all_conditions.csv"
        df_sum.to_csv(outsum_all, index=False, encoding="utf-8-sig")
        # 本文向け（cumeig のみ）
        df_main = df_sum[df_sum["mode"]=="cumeig"].copy()
        outsum_main = OUT_MAIN / "Table_4.2.4.4_summary_cumeig_only.csv"
        df_main.to_csv(outsum_main, index=False, encoding="utf-8-sig")

# ========= FP→OH（逆方向） =========
def build_reverse_direction():
    set_font()
    Xoh0 = load_features_bin(FEATURES_OH)
    Xfp0 = load_features_bin(FEATURES_FP)
    common = Xoh0.index.intersection(Xfp0.index)
    if len(common)==0:
        raise RuntimeError("No common samples between features_OH_onlyMat and features_FP_onlyMat.")
    Xoh0 = Xoh0.loc[common]; Xfp0 = Xfp0.loc[common]

    oh_latest = latest_by_mode_index(FIGS_OH, "OH")
    fp_latest = latest_by_mode_index(FIGS_FP, "FP")
    keys = [(m,i) for m in DIMS for i in INDICES if (m,i) in oh_latest and (m,i) in fp_latest]
    if not keys:
        raise RuntimeError("No common (mode,index) between figs_OH and figs_FP.")

    summary_rows = []

    for (mode,index) in keys:
        print(f"[REV] {mode} × {index}")
        oh_assign = read_var_cluster(oh_latest[(mode,index)])
        fp_assign = read_var_cluster(fp_latest[(mode,index)])

        # フラグメント → OH 主分布
        fragments = [f for f in fp_assign.index if f in Xfp0.columns]
        rows_frag=[]
        for frag in fragments:
            n_frag = int(Xfp0[frag].sum())
            if n_frag<=0: continue
            maj, purity, n_s, dist = fragment_to_oh_distribution(frag, Xfp0, Xoh0, oh_assign, threshold=MAT_PREV_THRESHOLD)
            fp_c = fp_assign.get(frag, np.nan)
            H = entropy_from_dist(dist); k_eff = len(dist) if dist else 0
            rows_frag.append({
                "mode":mode,"index":index,
                "fragment":frag,
                "FP_cluster": int(fp_c) if pd.notna(fp_c) else np.nan,
                "n_samples_with_fragment": n_s,
                "OH_major_cluster": maj,
                "OH_purity": purity,
                "OH_entropy": H,
                "OH_k_effective": k_eff,
                "OH_dist": ";".join(f"{k}:{v:.3f}" for k,v in sorted(dist.items())) if dist else ""
            })
        df_frag = pd.DataFrame(rows_frag)
        out_csv1 = OUT_R_CSV / f"Table_4.2.4.4R_fragment-to-OHmajor_{mode}_{index}.csv"
        if len(df_frag): df_frag.to_csv(out_csv1, index=False, encoding="utf-8-sig")

        # 同一FPクラスタ内ペアの材料側一貫性
        rows_pair=[]
        if len(df_frag):
            def parse_dist(s):
                if isinstance(s,str) and s:
                    d={}
                    for t in s.split(";"):
                        k,v = t.split(":"); d[int(k)] = float(v)
                    return d
                return {}
            dists = {r.fragment: parse_dist(r.OH_dist) for r in df_frag.itertuples()}
            all_oh_clusters = sorted({k for d in dists.values() for k in d.keys()})
            def vec_of(dist):
                return np.array([dist.get(k, 0.0) for k in all_oh_clusters], dtype=float)

            for k, grp in df_frag.dropna(subset=["FP_cluster"]).groupby("FP_cluster"):
                frags = grp["fragment"].tolist()
                for a,b in combinations(frags,2):
                    da, db = dists.get(a,{}), dists.get(b,{})
                    va, vb = vec_of(da), vec_of(db)
                    if va.sum()==0 and vb.sum()==0:
                        cos_sim=np.nan; jsd=np.nan
                    else:
                        cos_sim = 1.0 - (cosine(va, vb) if (va.sum()>0 and vb.sum()>0) else 1.0)
                        jsd = js_divergence(va, vb) if (va.sum()>0 and vb.sum()>0) else np.nan
                    maj_same = (df_frag.loc[df_frag["fragment"]==a,"OH_major_cluster"].values[0] ==
                                df_frag.loc[df_frag["fragment"]==b,"OH_major_cluster"].values[0])
                    purity_min = float(min(
                        df_frag.loc[df_frag["fragment"]==a,"OH_purity"].values[0] if len(df_frag.loc[df_frag["fragment"]==a,"OH_purity"]) else 0.0,
                        df_frag.loc[df_frag["fragment"]==b,"OH_purity"].values[0] if len(df_frag.loc[df_frag["fragment"]==b,"OH_purity"]) else 0.0
                    ))
                    rows_pair.append({
                        "mode":mode,"index":index,
                        "FP_cluster":int(k),
                        "fragment_A":a,"fragment_B":b,
                        "OH_major_same":bool(maj_same),
                        "OH_purity_min": purity_min,
                        "OH_cosine_similarity": cos_sim,
                        "OH_JS_divergence": jsd
                    })
        df_pair = pd.DataFrame(rows_pair)
        out_csv2 = OUT_R_CSV / f"Table_4.2.4.4R_fragment-pair_cohesion_{mode}_{index}.csv"
        if len(df_pair): df_pair.to_csv(out_csv2, index=False, encoding="utf-8-sig")

        if len(df_pair):
            def label_cohesion(r, purity_thr=PURITY_FLAG_THR, cos_thr=COS_SIM_THR, js_thr=JS_DIV_THR):
                return bool(
                    (r["OH_major_same"] and r["OH_purity_min"]>=purity_thr) or
                    (pd.notna(r["OH_cosine_similarity"]) and r["OH_cosine_similarity"]>=cos_thr) or
                    (pd.notna(r["OH_JS_divergence"]) and r["OH_JS_divergence"]<=js_thr)
                )
            df_pair["cohesive_flag"] = df_pair.apply(label_cohesion, axis=1)
            df_fpc = (df_pair.groupby(["mode","index","FP_cluster"], as_index=False)
                             .agg(n_pairs=("cohesive_flag","size"),
                                  n_cohesive=("cohesive_flag","sum")))
            df_fpc["cohesive_ratio"] = df_fpc["n_cohesive"]/df_fpc["n_pairs"].replace(0, np.nan)
        else:
            df_fpc = pd.DataFrame()
        out_csv3 = OUT_R_CSV / f"Table_4.2.4.4R_FPcluster_cohesion_{mode}_{index}.csv"
        if len(df_fpc): df_fpc.to_csv(out_csv3, index=False, encoding="utf-8-sig")

        # 図（本文/付録）
        is_main = (mode,index) in MAIN_PAIRS
        out_dir = (OUT_R_MAIN if is_main else OUT_R_APPENDIX) / f"FigSet_4.2.4.4R_{mode}_{index}"
        out_dir.mkdir(parents=True, exist_ok=True)
        plot_fragment_scatter(df_frag, out_dir, mode, index, top_labels=15, main_text=is_main)
        plot_pair_distributions_reverse(df_pair, out_dir, mode, index, main_text=is_main)
        plot_fpcluster_consistency(df_fpc, out_dir, mode, index, main_text=is_main)
        plot_mini_sankey_fragment_to_oh(df_frag, out_dir, mode, index, topN=20, main_text=is_main)

        # サマリー
        summary_rows.append({
            "mode":mode,"index":index,
            "n_fragments": len(df_frag),
            "mean_OH_purity": df_frag["OH_purity"].dropna().mean() if "OH_purity" in df_frag else np.nan,
            "mean_OH_entropy": df_frag["OH_entropy"].dropna().mean() if "OH_entropy" in df_frag else np.nan,
            "pair_OH_major_same_rate": df_pair["OH_major_same"].mean() if len(df_pair) else np.nan,
            "pair_mean_cosine_similarity": df_pair["OH_cosine_similarity"].dropna().mean() if len(df_pair) else np.nan,
            "pair_mean_JS_divergence": df_pair["OH_JS_divergence"].dropna().mean() if len(df_pair) else np.nan,
            "FPcluster_median_cohesive_ratio": df_fpc["cohesive_ratio"].median() if len(df_fpc) else np.nan
        })

    if summary_rows:
        df_sum = pd.DataFrame(summary_rows).sort_values(["mode","index"])
        outsum = OUT_R_ROOT / "Table_4.2.4.4R_summary_all_conditions.csv"
        df_sum.to_csv(outsum, index=False, encoding="utf-8-sig")

# ========= signless vs new: OH→FP 側の一致度（OH/FP個別） =========
def collect_assign_labels(figs_dir: Path, ds: str) -> pd.DataFrame | None:
    pat = re.compile(rf"^ClusterAssign_(top3|cumeig)_(silhouette|dunn|gap|ch|db|ptbiserial)_{ds}_.*\.csv$")
    files = [p for p in figs_dir.glob("ClusterAssign_*.csv") if pat.match(p.name)]
    parts=[]
    for f in files:
        m = re.match(r"ClusterAssign_(top3|cumeig)_([A-Za-z]+)_", f.name)
        if not m: continue
        mode, index = m.groups()
        df = read_csv_quiet(f)
        if df is None: continue
        if {"Variable","Cluster"}.issubset(df.columns):
            ids = df["Variable"].astype(str).values
            cl  = df["Cluster"].values
        elif df.shape[1]==1:
            ids = df.index.astype(str).values
            cl  = df.iloc[:,0].values
        else:
            continue
        codes = pd.Categorical(cl).codes
        codes = pd.Series(codes).replace({-1: np.nan}).values
        parts.append(pd.DataFrame({"id": ids, f"{mode}_{index}": codes}))
    if not parts: return None
    out = parts[0]
    for p in parts[1:]:
        out = out.merge(p, on="id", how="outer")
    return out.drop_duplicates(subset=["id"])

def load_signless_labels(signless_dir: Path) -> pd.DataFrame | None:
    cands = sorted(signless_dir.glob("cluster_labels_*.csv"), key=os.path.getmtime, reverse=True)
    if not cands: return None
    df = read_csv_quiet(cands[0])
    if df is None or "id" not in df.columns: return None
    keep = [c for c in df.columns if str(c).startswith("linear_")]
    if not keep: return None
    out = df[["id"]+keep].copy()
    out.columns = ["id"] + [c.replace("linear_","",1) for c in keep]
    return out

def pairwise_scores(dfA: pd.DataFrame|None, dfB: pd.DataFrame|None) -> pd.DataFrame:
    cols = ["condition","n","kA","kB","ARI","NMI","AMI"]
    if dfA is None or dfB is None: return pd.DataFrame(columns=cols)
    colsA = [c for c in dfA.columns if c!="id"]
    colsB = [c for c in dfB.columns if c!="id"]
    common = sorted(set(colsA).intersection(colsB))
    if not common: return pd.DataFrame(columns=cols)
    merged = dfA.merge(dfB, on="id", suffixes=(".A",".B"))
    rows=[]
    for cn in common:
        a = merged[f"{cn}.A"].values
        b = merged[f"{cn}.B"].values
        mask = ~pd.isna(a) & ~pd.isna(b)
        if mask.sum() < 2: continue
        a = a[mask].astype(int); b = b[mask].astype(int)
        rows.append([
            cn, int(mask.sum()),
            int(len(np.unique(a))), int(len(np.unique(b))),
            float(adjusted_rand_score(a,b)),
            float(normalized_mutual_info_score(a,b)),
            float(adjusted_mutual_info_score(a,b)),
        ])
    return pd.DataFrame(rows, columns=cols)

def lighten(color, factor=0.5):
    r,g,b = matplotlib.colors.to_rgb(color)
    return (1 - factor) + factor*r, (1 - factor) + factor*g, (1 - factor) + factor*b

def save_bar_contrast_with_k(df_long: pd.DataFrame, T_meta: pd.DataFrame, out_dir: Path, title: str, fname_core: str, fade_outside: bool):
    set_font()
    INDEX_LIST = ["silhouette","dunn","gap","ch","db","ptbiserial"]
    DATASETS = ["OH","FP"]

    df = df_long.copy()
    df["mode"]  = pd.Categorical(df["mode"], categories=["top3","cumeig"], ordered=True)
    df["index"] = pd.Categorical(df["index"], categories=INDEX_LIST, ordered=True)
    df = df.sort_values(["index","mode","Dataset"])

    # x slots
    x_labels = [f"{m}\n{ix}" for m in ["top3","cumeig"] for ix in INDEX_LIST]
    grid=[]
    for m in ["top3","cumeig"]:
        for ix in INDEX_LIST:
            sl = df[(df["mode"]==m) & (df["index"]==ix)]
            grid.append(sl.set_index("Dataset")["ARI"].reindex(DATASETS))
    X = pd.concat(grid, axis=1).T
    X.index = x_labels

    # META lookup
    M = T_meta.copy()
    M["row_key"] = M["mode"] + "\n" + M["index"]
    M["is_target"] = (M[["kA","kB"]].max(axis=1) <= K_MAX_KEEP)
    META = (M.set_index(["row_key","Dataset"])[["kA","kB","is_target"]]
              .reindex(pd.MultiIndex.from_product([x_labels, DATASETS], names=["row_key","Dataset"])))

    color_OH = "#1f77b4"; color_FP="#ff7f0e"
    fade_OH  = lighten(color_OH, 0.7); fade_FP = lighten(color_FP, 0.7)

    fig, ax = plt.subplots(figsize=(12,6), dpi=DPI)
    width = 0.38; x = np.arange(X.shape[0])

    for i, ds in enumerate(DATASETS):
        base_color = color_OH if ds=="OH" else color_FP
        base_fade  = fade_OH  if ds=="OH" else fade_FP
        xs = x + (i-0.5)*width
        vals = X[ds].values
        for xi, val, row_key in zip(xs, vals, X.index):
            if (row_key, ds) in META.index:
                kA = META.loc[(row_key, ds), "kA"]; kB = META.loc[(row_key, ds), "kB"]
                is_t = bool(META.loc[(row_key, ds), "is_target"])
            else:
                kA = np.nan; kB = np.nan; is_t=False
            fc = base_color if (not fade_outside or is_t) else base_fade
            alpha = 0.95 if (not fade_outside or is_t) else 0.5
            ax.bar(xi, val, width, color=fc, alpha=alpha, edgecolor="none")
            if not np.isnan(val):
                label = f"{val:.2f}\n({int(kA) if not np.isnan(kA) else '-'}" \
                        f"/{int(kB) if not np.isnan(kB) else '-'})"
                ax.text(xi, val + 0.015, label, ha="center", va="bottom", fontsize=8)

    ax.set_xticks(x); ax.set_xticklabels(X.index, rotation=35, ha="right")
    ax.set_ylim(0,1.05); ax.set_ylabel("ARI")
    ax.set_title(title)
    legend_elems = [Patch(facecolor=color_OH, edgecolor="none", alpha=0.95, label="OH (k≤30)"),
                    Patch(facecolor=color_FP, edgecolor="none", alpha=0.95, label="FP (k≤30)")]
    if fade_outside:
        legend_elems += [Patch(facecolor=fade_OH, edgecolor="none", alpha=0.50, label="OH (k>30)"),
                         Patch(facecolor=fade_FP, edgecolor="none", alpha=0.50, label="FP (k>30)")]
    ax.legend(handles=legend_elems, loc="upper left", bbox_to_anchor=(1.02, 1.0))
    ax.grid(axis="y", linestyle="--", linewidth=0.4, alpha=0.6)
    fig.tight_layout()
    out_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_dir / f"{fname_core}.png", bbox_inches="tight")
    fig.savefig(out_dir / f"{fname_core}.pdf", bbox_inches="tight")
    plt.close(fig)

def save_ari_bar_for_main(df_cond: pd.DataFrame, mode: str, out_dir: Path):
    # df_cond: columns に ["mode","index","ARI"] を含むこと
    set_font()
    df = df_cond[df_cond["mode"]==mode].sort_values("index")
    if df.empty:
        return
    fig, ax = plt.subplots(figsize=(8,4), dpi=DPI)
    ax.bar(df["index"], df["ARI"], alpha=0.95)
    for xi, r in enumerate(df.itertuples()):
        ax.text(xi, r.ARI+0.015, f"{r.ARI:.2f}", ha="center", va="bottom", fontsize=8)
    ax.set_ylim(0,1.05); ax.set_ylabel("ARI (sample-level)")
    ax.set_title(f"Fig 4.2.4.4: ARI by index — {mode}")
    ax.grid(axis="y", linestyle="--", linewidth=0.4, alpha=0.6)
    fig.tight_layout()
    out_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_dir / f"Fig_4.2.4.4_ARI_by-index_{mode}.png", bbox_inches="tight")
    fig.savefig(out_dir / f"Fig_4.2.4.4_ARI_by-index_{mode}.pdf", bbox_inches="tight")
    plt.close(fig)

    
def build_ohfp_contrast():
    # signless vs new を OH/FP で独立に算出し、本文（k≤30）/付録（allK）を出力
    all_tbl_main=[]; all_for_contrast_main=[]; all_tbl_allK=[]; all_for_contrast_allK=[]
    for ds, figs in [("OH",FIGS_OH), ("FP",FIGS_FP)]:
        A = collect_assign_labels(figs, ds)
        S = load_signless_labels(SIGNLESS_DIR[ds])
        T = pairwise_scores(A, S)
        if T.empty:
            print(f"[WARN] {ds}: no common conditions")
            continue
        T["mode"]    = T["condition"].str.replace(r"_(.*)$", "", regex=True)
        T["index"]   = T["condition"].str.replace(r"^(.*)_", "", regex=True)
        T["Dataset"] = ds
        T["k_max"]   = T[["kA","kB"]].max(axis=1)

        all_tbl_allK.append(T[["condition","Dataset","n","kA","kB","ARI","NMI","AMI","mode","index","k_max"]])
        all_for_contrast_allK.append(T[["Dataset","mode","index","ARI","kA","kB","k_max"]])

        T30 = T[(T["k_max"].isna()) | (T["k_max"]<=K_MAX_KEEP)].copy()
        if len(T30):
            all_tbl_main.append(T30[["condition","Dataset","n","kA","kB","ARI","NMI","AMI","mode","index","k_max"]])
            all_for_contrast_main.append(T30[["Dataset","mode","index","ARI","kA","kB","k_max"]])

    if all_tbl_main:
        T_all = pd.concat(all_tbl_main, ignore_index=True)
        out_csv = OUT_MAIN / "Table_4.2.4.4_OHFP-contrast_summary_k-le30.csv"
        T_all.sort_values(["Dataset","mode","index"]).to_csv(out_csv, index=False, encoding="utf-8-sig")
        print(f"[SAVE] {out_csv}")

        contrast_src = pd.concat(all_for_contrast_main, ignore_index=True)
        for mode,index in MAIN_PAIRS:
            sub = contrast_src[(contrast_src["mode"]==mode)&(contrast_src["index"]==index)]
            out_dir = OUT_MAIN / f"FigSet_4.2.4.4_{mode}_{index}"
            save_bar_contrast_with_k(
                df_long=sub,
                T_meta=T_all[(T_all["mode"]==mode) & (T_all["index"]==index)][["Dataset","mode","index","kA","kB","k_max"]],
                out_dir=out_dir,
                title=f"Fig 4.2.4.4: OH vs FP (ARI of new vs signless, {mode}×{index}, k≤30)",
                fname_core="Fig_4.2.4.4_OHFP-contrast_ARI-bars",
                fade_outside=False
            )
    else:
        print("[MAIN] No k≤30 rows for OH→FP contrast.")

    if all_tbl_allK:
        T_allK = pd.concat(all_tbl_allK, ignore_index=True)
        out_csv_apx = OUT_APPENDIX / "Appendix_Table_allK_OHFP-contrast_summary.csv"
        T_allK.sort_values(["Dataset","mode","index"]).to_csv(out_csv_apx, index=False, encoding="utf-8-sig")
        print(f"[SAVE] {out_csv_apx}")

        contrast_allK = pd.concat(all_for_contrast_allK, ignore_index=True)
        save_bar_contrast_with_k(
            df_long=contrast_allK[["Dataset","mode","index","ARI"]],
            T_meta=T_allK[["Dataset","mode","index","kA","kB","k_max"]],
            out_dir=OUT_APPENDIX,
            title="Appendix: OH vs FP (ARI of new vs signless, all k)",
            fname_core="Appendix_Fig_allK_OHFP-contrast_ARI-bars",
            fade_outside=True
        )

# ========= 双方向まとめ =========
def build_bidirectional_summary():
    fwd = OUT_OHFP_ROOT / "Table_4.2.4.4_summary_all_conditions.csv"
    rev = OUT_R_ROOT    / "Table_4.2.4.4R_summary_all_conditions.csv"
    if not fwd.exists() or not rev.exists():
        print("[INFO] Skip bidirectional summary (source tables not found).")
        return
    A = pd.read_csv(fwd); R = pd.read_csv(rev)

    # 前方向：各 (mode,index) の代表スコア（ここでは mean FP_purity などではなく signless ではないため簡略化）
    # 便宜上、前方向の“整合度スコア”として pair_FP_major_same_rate を採用（必要に応じて他の合成指標に変更可）
    fwd_sc = (A.groupby(["mode","index"])["pair_FP_major_same_rate"].mean().rename("forward_score")).reset_index()

    # 逆方向：複合スコア（正の指標は↑、負の指標は↓）
    def minmax(s):
        s2 = s.copy()
        if s2.isna().all(): return s2.fillna(0.5)
        s2 = s2.fillna(s2.mean()); mn, mx = s2.min(), s2.max()
        if np.isclose(mx, mn): return pd.Series(0.5, index=s2.index)
        return (s2 - mn) / (mx - mn)

    rev2 = R.copy()
    pos = [c for c in ["mean_OH_purity","pair_OH_major_same_rate","pair_mean_cosine_similarity","FPcluster_median_cohesive_ratio"] if c in rev2.columns]
    neg = [c for c in ["mean_OH_entropy","pair_mean_JS_divergence"] if c in rev2.columns]
    parts = []
    for c in pos: parts.append(minmax(rev2[c]))
    for c in neg: parts.append(1 - minmax(rev2[c]))
    if parts:
        rev2["reverse_score"] = pd.concat(parts, axis=1).mean(axis=1)
    else:
        rev2["reverse_score"] = np.nan
    rev_sc = rev2.groupby(["mode","index"])["reverse_score"].mean().reset_index()

    M = fwd_sc.merge(rev_sc, on=["mode","index"], how="outer").sort_values(["mode","index"])
    out_csv = OUT_BIDIR / "Table_4.2.4.4_bidirectional_summary.csv"
    M.to_csv(out_csv, index=False, encoding="utf-8-sig")
    print(f"[SAVE] {out_csv}")

    # 図
    set_font()
    fig, ax = plt.subplots(figsize=(10,5), dpi=DPI)
    lbls = [f"{m}-{i}" for m,i in zip(M["mode"], M["index"])]
    x = np.arange(len(M)); w = 0.38
    ax.bar(x-w/2, M["forward_score"].values, width=w, label="OH→FP (proxy score)", alpha=0.9)
    ax.bar(x+w/2, M["reverse_score"].values, width=w, label="FP→OH (composite)", alpha=0.9)
    ax.set_xticks(x); ax.set_xticklabels(lbls, rotation=35, ha="right")
    ax.set_ylim(0,1.05); ax.set_ylabel("Score (0–1)")
    ax.set_title("Fig 4.2.4.4: Bidirectional alignment summary")
    ax.legend(); ax.grid(axis="y", linestyle="--", linewidth=0.4, alpha=0.6)
    fig.tight_layout()
    fig.savefig(OUT_BIDIR / "Fig_4.2.4.4_bidirectional_comparison.png", bbox_inches="tight")
    fig.savefig(OUT_BIDIR / "Fig_4.2.4.4_bidirectional_comparison.pdf", bbox_inches="tight")
    plt.close(fig)

# ========= （任意）k固定 Permutation / Bootstrap 検証 =========
def permutation_pvalue(yA: np.ndarray, yB: np.ndarray, n_perm=N_PERM):
    obs = adjusted_rand_score(yA, yB)
    null = np.zeros(n_perm, dtype=float)
    for i in range(n_perm):
        perm = rng.permutation(yB)
        null[i] = adjusted_rand_score(yA, perm)
    p = float((np.sum(null >= obs) + 1) / (n_perm + 1))  # 右片側
    return obs, null, p

def bootstrap_ci(yA: np.ndarray, yB: np.ndarray, n_boot=N_BOOT, alpha=0.05):
    n = len(yA)
    vals = np.zeros(n_boot, dtype=float)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        vals[i] = adjusted_rand_score(yA[idx], yB[idx])
    lo = float(np.quantile(vals, alpha/2)); hi = float(np.quantile(vals, 1-alpha/2))
    return (lo, hi), vals

def plot_perm_hist(null_vals: np.ndarray, obs_ari: float, mode: str, index: str, outdir: Path):
    set_font()
    fig, ax = plt.subplots(figsize=(7,4), dpi=DPI)
    ax.hist(null_vals, bins=30, alpha=0.9)
    ax.axvline(obs_ari, color="red", linestyle="--", linewidth=1.5, label=f"Observed ARI={obs_ari:.3f}")
    ax.set_title(f"Fig 4.2.4.4: Permutation null vs observed — {mode} × {index}")
    ax.set_xlabel("ARI under label permutation (null)"); ax.set_ylabel("Count")
    ax.legend(); fig.tight_layout()
    fig.savefig(outdir / f"Fig_4.2.4.4_perm-hist_{mode}_{index}.png", bbox_inches="tight")
    fig.savefig(outdir / f"Fig_4.2.4.4_perm-hist_{mode}_{index}.pdf", bbox_inches="tight")
    plt.close(fig)

def plot_kfixed_bars(df: pd.DataFrame, mode: str, k_filter: str, outdir: Path):
    if df.empty: return
    set_font()
    df = df.sort_values("index")
    fig, ax = plt.subplots(figsize=(9,4), dpi=DPI)
    x = np.arange(len(df))
    colors = ["#4E79A7" if p<=0.05 else "#9FA8B3" for p in df["p_perm"]]
    ax.bar(x, df["ARI"], color=colors, alpha=0.95)
    for xi, r in enumerate(df.itertuples()):
        ax.text(xi, r.ARI+0.01, f"{r.ARI:.2f}\np={r.p_perm:.3f}", ha="center", va="bottom", fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(df["index"], rotation=35, ha="right")
    ax.set_ylim(0,1.05); ax.set_ylabel("ARI (OH dominant vs FP dominant)")
    ax.set_title(f"Fig 4.2.4.4: ARI by index — {mode} ({'kOH=kFP=2' if k_filter=='k2_only' else 'k≠2含む'})")
    ax.grid(axis="y", linestyle="--", linewidth=0.4, alpha=0.6)
    fname_core = f"Fig_4.2.4.4_{k_filter}_ARIs_by-index_{mode}"
    fig.tight_layout()
    fig.savefig(outdir / f"{fname_core}.png", bbox_inches="tight")
    fig.savefig(outdir / f"{fname_core}.pdf", bbox_inches="tight")
    plt.close(fig)

def build_validation_kfixed():
    set_font()
    Xoh0 = load_features_bin(FEATURES_OH)
    Xfp0 = load_features_bin(FEATURES_FP)
    common_samples = Xoh0.index.intersection(Xfp0.index)
    if len(common_samples) == 0:
        print("[VALID] Skip: no common samples.")
        return
    Xoh0 = Xoh0.loc[common_samples]; Xfp0 = Xfp0.loc[common_samples]

    oh_latest = latest_by_mode_index(FIGS_OH, "OH")
    fp_latest = latest_by_mode_index(FIGS_FP, "FP")
    keys = [(m,i) for m in DIMS for i in INDICES if (m,i) in oh_latest and (m,i) in fp_latest]
    if not keys:
        print("[VALID] Skip: no common (mode,index)."); return

    rows_cond = []

    for (mode, index) in keys:
        oh_assign = read_var_cluster(oh_latest[(mode,index)])
        fp_assign = read_var_cluster(fp_latest[(mode,index)])

        Xoh = Xoh0[[c for c in Xoh0.columns if c in oh_assign.index]]
        Xfp = Xfp0[[c for c in Xfp0.columns if c in fp_assign.index]]
        lab_oh = derive_sample_labels(Xoh, oh_assign, min_votes=1)
        lab_fp = derive_sample_labels(Xfp, fp_assign, min_votes=1)

        mask = lab_oh.notna() & lab_fp.notna()
        if mask.sum() < 5:  # 少数例回避
            continue
        yA = lab_oh[mask].astype(int).values
        yB = lab_fp[mask].astype(int).values

        # 観測
        ari_obs = float(adjusted_rand_score(yA, yB))
        k_oh = int(pd.Series(oh_assign.dropna().astype(str).astype(int)).nunique())
        k_fp = int(pd.Series(fp_assign.dropna().astype(str).astype(int)).nunique())
        n_eff = int(mask.sum())

        # 置換検定・ブートCI
        ari_obs2, null_vals, p_perm = permutation_pvalue(yA, yB, n_perm=N_PERM)
        (ci_lo, ci_hi), boot_vals = bootstrap_ci(yA, yB, n_boot=N_BOOT, alpha=0.05)

        # 保存
        rows_cond.append({
            "mode": mode, "index": index, "n_samples": n_eff, "k_OH": k_oh, "k_FP": k_fp,
            "ARI": ari_obs, "p_perm": p_perm, "boot_CI_lo": ci_lo, "boot_CI_hi": ci_hi
        })

        # 図：置換ヒスト
        plot_perm_hist(null_vals, ari_obs, mode, index, OUT_VALID)

    # 条件別テーブル
    df_cond = pd.DataFrame(rows_cond).sort_values(["mode","index"])
    out_csv_cond = OUT_VALID / "Table_4.2.4.4_validation_per-condition.csv"
    df_cond.to_csv(out_csv_cond, index=False, encoding="utf-8-sig")
    print(f"[SAVE] {out_csv_cond}")

    # k固定（k=2 と それ以外）集計と図
    def summarize_block(df):
        if df.empty:
            return pd.DataFrame(columns=["mode","index","n_conditions","mean_ARI","median_ARI","sig_rate"])
        g = (df.groupby(["mode","index"], as_index=False)
               .agg(n_conditions=("ARI","size"),
                    mean_ARI=("ARI","mean"),
                    median_ARI=("ARI","median"),
                    sig_rate=("p_perm", lambda s: float((s<=0.05).mean()))))
        return g

    df_k2     = df_cond[(df_cond["k_OH"]==2) & (df_cond["k_FP"]==2)].copy()
    df_kNot2  = df_cond[~((df_cond["k_OH"]==2) & (df_cond["k_FP"]==2))].copy()
    sum_k2    = summarize_block(df_k2);    sum_k2["k_stratum"]    = "kOH=kFP=2"
    sum_kNot2 = summarize_block(df_kNot2); sum_kNot2["k_stratum"] = "kOH/kFP≠2含む"
    df_kfix   = pd.concat([sum_k2, sum_kNot2], ignore_index=True)
    out_csv_kfix = OUT_VALID / "Table_4.2.4.4_validation_k-fixed_summary.csv"
    df_kfix.to_csv(out_csv_kfix, index=False, encoding="utf-8-sig")
    print(f"[SAVE] {out_csv_kfix}")

    for mode in DIMS:
        plot_kfixed_bars(df_k2[df_k2["mode"]==mode][["index","ARI","p_perm"]], mode, "k2_only", OUT_VALID)
        plot_kfixed_bars(df_kNot2[df_kNot2["mode"]==mode][["index","ARI","p_perm"]], mode, "kNot2", OUT_VALID)
       
        # Fig.(a) を明示保存：本文は cumeig、付録は top3
        save_ari_bar_for_main(df_cond, "cumeig", OUT_MAIN)
        save_ari_bar_for_main(df_cond, "top3",  OUT_APPENDIX)

# ========= MAIN =========
def main(run_forward=True, run_reverse=True, run_contrast=True, run_bidir=True, run_validation=True):
    set_font()
    if run_contrast:
        print("=== [1/5] OH↔FP (new vs signless) contrast ===")
        build_ohfp_contrast()
    if run_forward:
        print("=== [2/5] OH→FP forward direction ===")
        build_forward_direction()
    if run_reverse:
        print("=== [3/5] FP→OH reverse direction ===")
        build_reverse_direction()
    if run_bidir:
        print("=== [4/5] Bidirectional summary ===")
        build_bidirectional_summary()
    if run_validation:
        print("=== [5/5] k-fixed validation (permutation & bootstrap) ===")
        build_validation_kfixed()
    print("\n✅ Done. Outputs under:", OUT_OHFP_ROOT)

if __name__ == "__main__":
    main(
        run_forward=True,     # Fig(b)(c)(d) 前方向の材料散布・ペア分布・OH cohesion
        run_reverse=True,     # 逆方向（付録・双方向整合に必要なら）
        run_contrast=False,   # 4.2.4.3関連は付録へ
        run_bidir=True,       # 双方向サマリ図
        run_validation=True   # Fig(a) ARI + k固定検証
    )


=== [1/5] OH↔FP (new vs signless) contrast ===
[WARN] OH: no common conditions
[SAVE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP_20251016_comp/main_text/Table_4.2.4.4_OHFP-contrast_summary_k-le30.csv
[SAVE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP_20251016_comp/appendix/Appendix_Table_allK_OHFP-contrast_summary.csv
=== [2/5] OH→FP forward direction ===
[FWD] top3 × silhouette
[FWD] top3 × dunn
[FWD] top3 × gap
[FWD] top3 × ch
[FWD] top3 × db
[FWD] top3 × ptbiserial
[FWD] cumeig × silhouette
[FWD] cumeig × dunn
[FWD] cumeig × gap
[FWD] cumeig × ch
[FWD] cumeig × db
[FWD] cumeig × ptbiserial
=== [3/5] FP→OH reverse direction ===
[REV] top3 × silhouette
[REV] top3 × dunn
[REV] top3 × gap
[REV] top3 × ch
[REV] top3 × db
[REV] top3 × ptbiserial
[REV] cumeig × silhouette
[REV] cumeig × dunn
[REV] cumeig × gap
[REV] cumeig × ch
[REV] cumeig × db
[REV] cumeig × ptbiserial
=== [4/5] Bidirectional summary ===
[SAVE] /Volum

/tmp/ipykernel_3946702/1337746206.py:1024: UserWarning: Glyph 21547 (\N{CJK UNIFIED IDEOGRAPH-542B}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/tmp/ipykernel_3946702/1337746206.py:1024: UserWarning: Glyph 12416 (\N{HIRAGANA LETTER MU}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/tmp/ipykernel_3946702/1337746206.py:1025: UserWarning: Glyph 21547 (\N{CJK UNIFIED IDEOGRAPH-542B}) missing from font(s) DejaVu Sans.
  fig.savefig(outdir / f"{fname_core}.png", bbox_inches="tight")
/tmp/ipykernel_3946702/1337746206.py:1025: UserWarning: Glyph 12416 (\N{HIRAGANA LETTER MU}) missing from font(s) DejaVu Sans.
  fig.savefig(outdir / f"{fname_core}.png", bbox_inches="tight")
/tmp/ipykernel_3946702/1337746206.py:1026: UserWarning: Glyph 21547 (\N{CJK UNIFIED IDEOGRAPH-542B}) missing from font(s) DejaVu Sans.
  fig.savefig(outdir / f"{fname_core}.pdf", bbox_inches="tight")
/tmp/ipykernel_3946702/1337746206.py:1026: UserWarning: Glyph 12416 (\N{HIRAGANA LETTER MU}) missing fro


✅ Done. Outputs under: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP_20251016_comp


/tmp/ipykernel_3946702/1337746206.py:1026: UserWarning: Glyph 21547 (\N{CJK UNIFIED IDEOGRAPH-542B}) missing from font(s) DejaVu Sans.
  fig.savefig(outdir / f"{fname_core}.pdf", bbox_inches="tight")
/tmp/ipykernel_3946702/1337746206.py:1026: UserWarning: Glyph 12416 (\N{HIRAGANA LETTER MU}) missing from font(s) DejaVu Sans.
  fig.savefig(outdir / f"{fname_core}.pdf", bbox_inches="tight")


In [ ]:

# #!/usr/bin/env python3
# # -*- coding: utf-8 -*-

# """
# paper_4.2.4.4_all_build_integrated.py
# -------------------------------------
# 4.2.4.4節（OH系とFP系のクラスタ対応関係）の図表を“一括で”再現・比較・検証する統合スクリプト。

# 含まれる機能（元コードとの対応）:
#   (A/F/D) OH→FP 主分布・ペア一貫性・OHクラスタ一貫性・図表（本文/付録仕分け）・サマリー
#   (B/E)   FP→OH（逆方向）主分布・ペア一貫性・FPクラスタ一貫性・図表（本文/付録仕分け）・サマリー
#   (E)     signless と new の一致度（ARI/NMI/AMI）: OH/FP それぞれで集計し本文/付録図表を出力
#   (E)     双方向サマリー（OH→FPのスコア vs FP→OHの複合スコア）の表・図
#   (C)     k固定検証（Permutation 検定・Bootstrap CI・k=2限定/その他での集約と図）※任意実行

# 出力ルート（分かりやすさを重視した構成）:
#   ROOT/paper_4.2.4.4_OHFP/
#     ├─ main_text/                 ← 本文用（cumeig×gap / cumeig×db を中心）
#     ├─ appendix/                  ← 付録（その他条件・全Kなど）
#     ├─ analysis_csv/              ← OH→FP 前方向のCSV
#     ├─ reverse/
#     │    ├─ main_text/            ← 逆方向の本文図
#     │    ├─ appendix/             ← 逆方向の付録図
#     │    └─ analysis_csv/         ← 逆方向のCSV
#     ├─ bidirectional_summary/     ← 双方向まとめ
#     └─ validation/                ← k固定・Permutation・Bootstrap（任意）

# ※ 実行前に ROOT を環境に合わせて調整してください。
# """

# from pathlib import Path
# import os, re
# import numpy as np
# import pandas as pd
# import matplotlib
# import matplotlib.pyplot as plt
# from matplotlib import font_manager as fm
# from matplotlib.patches import Patch
# from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, adjusted_mutual_info_score
# from itertools import combinations
# from scipy.spatial.distance import cosine

# # ========= ユーザー設定 =========
# ROOT = Path("/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No")
# FIGS_OH = ROOT / "figs_OH"
# FIGS_FP = ROOT / "figs_FP"

# # signless 側（必要に応じて調整）
# SIGNLESS_BASE = ROOT / "STEP3.2_signlessCorr_MDS_20250904"
# SIGNLESS_DIR = {"OH": SIGNLESS_BASE / "OH", "FP": SIGNLESS_BASE / "FP"}

# # features（材料列のみ）
# FEATURES_OH = ROOT / "features_OH_onlyMat.csv"
# FEATURES_FP = ROOT / "features_FP_onlyMat.csv"

# # 出力ルート
# OUT_OHFP_ROOT   = ROOT / "paper_4.2.4.4_OHFP_20251016_comp"
# OUT_MAIN        = OUT_OHFP_ROOT / "main_text"
# OUT_APPENDIX    = OUT_OHFP_ROOT / "appendix"
# OUT_VARMAP      = OUT_OHFP_ROOT / "analysis_csv"
# OUT_R_ROOT      = OUT_OHFP_ROOT / "reverse"
# OUT_R_MAIN      = OUT_R_ROOT / "main_text"
# OUT_R_APPENDIX  = OUT_R_ROOT / "appendix"
# OUT_R_CSV       = OUT_R_ROOT / "analysis_csv"
# OUT_BIDIR       = OUT_OHFP_ROOT / "bidirectional_summary"
# OUT_VALID       = OUT_OHFP_ROOT / "validation"
# for d in [OUT_MAIN, OUT_APPENDIX, OUT_VARMAP, OUT_R_MAIN, OUT_R_APPENDIX, OUT_R_CSV, OUT_BIDIR, OUT_VALID]:
#     d.mkdir(parents=True, exist_ok=True)

# # 対象の mode/index
# DIMS    = ["top3","cumeig"]
# INDICES = ["silhouette","dunn","gap","ch","db","ptbiserial"]
# MAIN_PAIRS = {("cumeig","gap"), ("cumeig","db")}  # 本文優先

# # しきい値／定数
# FRAG_PREV_THRESHOLD = 0.20   # P(fragment=1 | material=1) 下限（前方向）
# MAT_PREV_THRESHOLD  = 0.20   # P(material=1 | fragment=1) 下限（逆方向）
# PURITY_FLAG_THR     = 0.60
# COS_SIM_THR         = 0.60
# JS_DIV_THR          = 0.50
# K_MAX_KEEP          = 30
# DPI                 = 300

# # 検証（任意実行）の反復数
# N_PERM = 500
# N_BOOT = 500
# RANDOM_SEED = 42
# rng = np.random.default_rng(RANDOM_SEED)

# # ========= フォント =========
# def set_font():
#     cand = ["Noto Sans CJK JP","Noto Sans JP","Source Han Sans JP","IPAexGothic","Yu Gothic","Meiryo","Hiragino Sans","Meiryo UI"]
#     have = set()
#     for p in fm.findSystemFonts(fontext="ttf"):
#         try:
#             have.add(fm.FontProperties(fname=p).get_name())
#         except:
#             pass
#     for w in cand:
#         if any(w.lower() in h.lower() for h in have):
#             matplotlib.rcParams["font.family"] = w
#             break
#     matplotlib.rcParams["axes.unicode_minus"] = False
#     matplotlib.rcParams.update({"font.size":10, "axes.titlesize":12, "axes.labelsize":10, "legend.fontsize":9})

# # ========= IO =========
# def read_csv_guess(path: Path) -> pd.DataFrame:
#     if not path or not path.exists():
#         raise FileNotFoundError(path)
#     for enc in ("utf-8","utf-8-sig","cp932"):
#         try:
#             return pd.read_csv(path, encoding=enc)
#         except Exception:
#             continue
#     raise RuntimeError(f"Failed to read: {path}")

# def read_csv_quiet(path: Path) -> pd.DataFrame | None:
#     if not path or not path.exists():
#         return None
#     for enc in ("utf-8","utf-8-sig","cp932"):
#         try:
#             return pd.read_csv(path, encoding=enc)
#         except Exception:
#             continue
#     return None

# def load_features_bin(path: Path) -> pd.DataFrame:
#     df = read_csv_guess(path)
#     if df.columns[0].lower() != "sample_id":
#         df = df.rename(columns={df.columns[0]:"sample_id"})
#     X = df.set_index("sample_id").apply(pd.to_numeric, errors="coerce").fillna(0.0)
#     return (X != 0).astype(int)

# # ========= 既存 ClusterAssign の最新検出 =========
# FN_RE = re.compile(
#     r"^ClusterAssign_(?P<mode>top3|cumeig)_(?P<index>silhouette|dunn|gap|ch|db|ptbiserial)_(?P<ds>OH|FP)_(?P<date>\d{8})_(?P<hm>\d{4})\.csv$"
# )

# def latest_by_mode_index(fig_dir: Path, ds: str):
#     latest = {}
#     for p in fig_dir.glob("ClusterAssign_*.csv"):
#         m = FN_RE.match(p.name)
#         if not m or m["ds"] != ds:
#             continue
#         key = (m["mode"], m["index"])
#         ts  = f"{m['date']}_{m['hm']}"
#         if key not in latest or ts > latest[key][0]:
#             latest[key] = (ts, p)
#     return {k: v[1] for k,v in latest.items()}

# def read_var_cluster(path: Path) -> pd.Series:
#     df = read_csv_guess(path)
#     cols = {c.lower(): c for c in df.columns}
#     vcol = cols.get("variable", df.columns[0])
#     ccol = cols.get("cluster",  df.columns[-1])
#     s = pd.Series(df[ccol].values, index=df[vcol].astype(str).values, name="cluster")
#     try:
#         s = s.astype(int)
#     except:
#         pass
#     return s

# # ========= 指標関数 =========
# def entropy_from_dist(dist: dict[int, float]) -> float:
#     if not dist:
#         return np.nan
#     p = np.array(list(dist.values()), dtype=float)
#     p = p / (p.sum() + 1e-12)
#     p = p[p > 0]
#     return float(-(p * np.log2(p)).sum())

# def js_divergence(p, q):
#     p = np.asarray(p, dtype=float); p = p/p.sum() if p.sum()>0 else p
#     q = np.asarray(q, dtype=float); q = q/q.sum() if q.sum()>0 else q
#     m = 0.5*(p+q)
#     def _kl(a,b):
#         a = np.clip(a,1e-12,1); b = np.clip(b,1e-12,1)
#         return float(np.sum(a*(np.log2(a)-np.log2(b))))
#     return 0.5*_kl(p,m) + 0.5*_kl(q,m)

# # ========= サンプル優勢クラスタ（投票） =========
# def derive_sample_labels(features: pd.DataFrame, var2clu: pd.Series, min_votes=1) -> pd.Series:
#     common = [v for v in features.columns if v in var2clu.index]
#     if not common:
#         return pd.Series(index=features.index, dtype="float64")
#     clus = pd.Series(var2clu.loc[common]).astype(str)
#     uniq = sorted(clus.unique())
#     mat = features[common].values
#     key2idx = {k:i for i,k in enumerate(uniq)}
#     col2cidx = np.array([key2idx[c] for c in clus.values], dtype=int)
#     votes = np.zeros((features.shape[0], len(uniq)), dtype=int)
#     rows, cols = np.where(mat==1)
#     for r, c in zip(rows, cols):
#         votes[r, col2cidx[c]] += 1
#     maxv = votes.max(axis=1)
#     labels = np.where(maxv>=min_votes, votes.argmax(axis=1), -1)
#     out = pd.Series(labels, index=features.index).replace({-1: np.nan})
#     lbl_vals = [uniq[int(i)] if pd.notna(i) else np.nan for i in out.values]
#     try:
#         return pd.Series(lbl_vals, index=features.index).astype(int)
#     except:
#         return pd.Series(lbl_vals, index=features.index)

# # ========= 前方向（OH→FP）: material → FP 分布 =========
# def material_to_fp_distribution(material, Xoh, Xfp, fp_var2clu, threshold=FRAG_PREV_THRESHOLD):
#     idx = Xoh.index[Xoh[material] == 1]
#     if len(idx) == 0:
#         return None, None, 0, {}
#     prev = Xfp.loc[idx].mean(axis=0)  # P(fragment=1 | material=1)
#     sel = prev[prev >= threshold]
#     if sel.empty:
#         return None, None, len(idx), {}
#     df = pd.DataFrame({"frag": sel.index, "w": sel.values})
#     df["fp_cluster"] = df["frag"].map(fp_var2clu).astype("Int64")
#     df = df.dropna(subset=["fp_cluster"])
#     if df.empty:
#         return None, None, len(idx), {}
#     grp = df.groupby("fp_cluster", as_index=False)["w"].sum()
#     grp["p"] = grp["w"]/grp["w"].sum()
#     dist = dict(zip(grp["fp_cluster"].astype(int), grp["p"]))
#     maj = int(grp.sort_values("p", ascending=False).iloc[0]["fp_cluster"])
#     purity = float(grp["p"].max())
#     return maj, purity, len(idx), dist

# # ========= 逆方向（FP→OH）: fragment → OH 分布 =========
# def fragment_to_oh_distribution(fragment, Xfp, Xoh, oh_var2clu, threshold=MAT_PREV_THRESHOLD):
#     idx = Xfp.index[Xfp[fragment] == 1]
#     if len(idx) == 0:
#         return None, None, 0, {}
#     prev = Xoh.loc[idx].mean(axis=0)  # P(material=1 | fragment=1)
#     sel = prev[prev >= threshold]
#     if sel.empty:
#         return None, None, len(idx), {}
#     df = pd.DataFrame({"mat": sel.index, "w": sel.values})
#     df["oh_cluster"] = df["mat"].map(oh_var2clu).astype("Int64")
#     df = df.dropna(subset=["oh_cluster"])
#     if df.empty:
#         return None, None, len(idx), {}
#     grp = df.groupby("oh_cluster", as_index=False)["w"].sum()
#     grp["p"] = grp["w"]/grp["w"].sum()
#     dist = dict(zip(grp["oh_cluster"].astype(int), grp["p"]))
#     maj = int(grp.sort_values("p", ascending=False).iloc[0]["oh_cluster"])
#     purity = float(grp["p"].max())
#     return maj, purity, len(idx), dist

# # ========= 図（前方向/逆方向で再利用） =========
# def _plot_pair_hist(ax, series: pd.Series, title: str, xlabel: str):
#     s = series.dropna()
#     if len(s):
#         ax.hist(s, bins=20, alpha=0.9)
#         ax.set_title(title); ax.set_xlabel(xlabel); ax.set_ylabel("Count")

# def plot_material_scatter(df_mat: pd.DataFrame, outdir: Path, mode: str, index: str, top_labels:int=15, main_text:bool=False):
#     set_font()
#     if df_mat is None or df_mat.empty: return
#     size = 30 + 3 * df_mat["n_samples_with_material"].fillna(0).astype(float)
#     c    = df_mat["OH_cluster"].astype("category")
#     fig, ax = plt.subplots(figsize=(9,6), dpi=DPI)
#     ax.scatter(df_mat["FP_entropy"], df_mat["FP_purity"], c=c.cat.codes, s=size, cmap="tab20", alpha=0.85, edgecolors="none")
#     ax.set_xlabel("FP entropy (larger = more fragmented)")
#     ax.set_ylabel("FP purity (larger = more cohesive)")
#     title = f"Fig 4.2.4.4: Materials — Cohesion vs Fragmentation ({mode} × {index})"
#     if main_text: title += " [Main]"
#     ax.set_title(title)
#     lab_df = df_mat.sort_values(["FP_purity","n_samples_with_material"], ascending=False).head(top_labels)
#     for _, r in lab_df.iterrows():
#         ax.text(r["FP_entropy"], r["FP_purity"]+0.015, str(r["material"]), ha="center", va="bottom", fontsize=8)
#     ax.grid(True, linestyle="--", linewidth=0.4, alpha=0.6)
#     outdir.mkdir(parents=True, exist_ok=True)
#     fig.tight_layout()
#     fname = f"Fig_4.2.4.4_materials_entropy-vs-purity_{mode}_{index}{'_MAIN' if main_text else ''}"
#     fig.savefig(outdir / f"{fname}.png", bbox_inches="tight"); fig.savefig(outdir / f"{fname}.pdf", bbox_inches="tight")
#     plt.close(fig)
    
# def plot_fragment_scatter(df_frag: pd.DataFrame, outdir: Path, mode: str, index: str, top_labels:int=15, main_text:bool=False):
#     set_font()
#     if df_frag is None or df_frag.empty: 
#         return
#     size = 30 + 3 * df_frag["n_samples_with_fragment"].fillna(0).astype(float)
#     c    = df_frag["FP_cluster"].astype("category")
#     fig, ax = plt.subplots(figsize=(9,6), dpi=DPI)
#     sc = ax.scatter(
#         df_frag["OH_entropy"], 
#         df_frag["OH_purity"], 
#         c=c.cat.codes, s=size, cmap="tab20", alpha=0.85, edgecolors="none"
#     )
#     ax.set_xlabel("OH entropy (larger = more materials-spread)")
#     ax.set_ylabel("OH purity (larger = more materials-specific)")
#     title = f"Fig 4.2.4.4R: Fragments — Specificity vs Spread ({mode} × {index})"
#     if main_text: 
#         title += " [Main]"
#     ax.set_title(title)

#     # ラベルは上位だけ
#     lab_df = df_frag.sort_values(
#         ["OH_purity","n_samples_with_fragment"], ascending=False
#     ).head(top_labels)
#     for _, r in lab_df.iterrows():
#         ax.text(
#             r["OH_entropy"], r["OH_purity"]+0.015, 
#             str(r["fragment"]), ha="center", va="bottom", fontsize=8
#         )

#     ax.grid(True, linestyle="--", linewidth=0.4, alpha=0.6)
#     outdir.mkdir(parents=True, exist_ok=True)
#     fig.tight_layout()
#     fname = f"Fig_4.2.4.4R_fragments_specificity-vs-spread_{mode}_{index}{'_MAIN' if main_text else ''}"
#     fig.savefig(outdir / f"{fname}.png", bbox_inches="tight")
#     fig.savefig(outdir / f"{fname}.pdf", bbox_inches="tight")
#     plt.close(fig)

# def plot_pair_distributions_forward(df_pair: pd.DataFrame, outdir: Path, mode: str, index: str, main_text:bool=False):
#     if df_pair is None or df_pair.empty: return
#     set_font()
#     fig, ax = plt.subplots(figsize=(8,4), dpi=DPI)
#     rate = df_pair["FP_major_same"].value_counts(normalize=True).rename({True:"Same", False:"Different"})
#     idx = ["Same","Different"]; vals = [rate.get("Same",0.0), rate.get("Different",0.0)]
#     ax.bar(idx, vals, color=["#4E79A7","#E15759"], alpha=0.9)
#     ax.set_ylim(0,1); ax.set_ylabel("Proportion")
#     title = f"Fig 4.2.4.4: Pair-level — FP major same? ({mode} × {index})"
#     if main_text: title += " [Main]"
#     ax.set_title(title)
#     for xi, val in enumerate(vals): ax.text(xi, val+0.02, f"{val:.2f}", ha="center", va="bottom")
#     fig.tight_layout()
#     fname = f"Fig_4.2.4.4_pair_major-same_{mode}_{index}{'_MAIN' if main_text else ''}"
#     fig.savefig(outdir / f"{fname}.png", bbox_inches="tight"); fig.savefig(outdir / f"{fname}.pdf", bbox_inches="tight")
#     plt.close(fig)

#     for col, xlabel in [("FP_cosine_similarity","Cosine similarity (1=similar)"),
#                         ("FP_JS_divergence","JS divergence (0=similar)")]:
#         s = df_pair[col].dropna()
#         if not len(s): continue
#         fig, ax = plt.subplots(figsize=(8,4), dpi=DPI)
#         _plot_pair_hist(ax, s, f"Fig 4.2.4.4: Pair-level — {col} ({mode} × {index})" + (" [Main]" if main_text else ""), xlabel)
#         fig.tight_layout()
#         fname = f"Fig_4.2.4.4_pair_{col}_hist_{mode}_{index}{'_MAIN' if main_text else ''}"
#         fig.savefig(outdir / f"{fname}.png", bbox_inches="tight"); fig.savefig(outdir / f"{fname}.pdf", bbox_inches="tight")
#         plt.close(fig)

# def plot_pair_distributions_reverse(df_pair: pd.DataFrame, outdir: Path, mode: str, index: str, main_text:bool=False):
#     if df_pair is None or df_pair.empty: return
#     set_font()
#     fig, ax = plt.subplots(figsize=(8,4), dpi=DPI)
#     rate = df_pair["OH_major_same"].value_counts(normalize=True).rename({True:"Same", False:"Different"})
#     idx = ["Same","Different"]; vals = [rate.get("Same",0.0), rate.get("Different",0.0)]
#     ax.bar(idx, vals, color=["#4E79A7","#E15759"], alpha=0.9)
#     ax.set_ylim(0,1); ax.set_ylabel("Proportion")
#     title = f"Fig 4.2.4.4R: Pair-level — OH major same? ({mode} × {index})"
#     if main_text: title += " [Main]"
#     ax.set_title(title)
#     for xi, val in enumerate(vals): ax.text(xi, val+0.02, f"{val:.2f}", ha="center", va="bottom")
#     fig.tight_layout()
#     fname = f"Fig_4.2.4.4R_pair_major-same_{mode}_{index}{'_MAIN' if main_text else ''}"
#     fig.savefig(outdir / f"{fname}.png", bbox_inches="tight"); fig.savefig(outdir / f"{fname}.pdf", bbox_inches="tight")
#     plt.close(fig)

#     for col, xlabel in [("OH_cosine_similarity","Cosine similarity (1=similar)"),
#                         ("OH_JS_divergence","JS divergence (0=similar)")]:
#         s = df_pair[col].dropna()
#         if not len(s): continue
#         fig, ax = plt.subplots(figsize=(8,4), dpi=DPI)
#         _plot_pair_hist(ax, s, f"Fig 4.2.4.4R: Pair-level — {col} ({mode} × {index})" + (" [Main]" if main_text else ""), xlabel)
#         fig.tight_layout()
#         fname = f"Fig_4.2.4.4R_pair_{col}_hist_{mode}_{index}{'_MAIN' if main_text else ''}"
#         fig.savefig(outdir / f"{fname}.png", bbox_inches="tight"); fig.savefig(outdir / f"{fname}.pdf", bbox_inches="tight")
#         plt.close(fig)

# def plot_ohcluster_consistency(df_ohc: pd.DataFrame, outdir: Path, mode: str, index: str, main_text:bool=False):
#     if df_ohc is None or df_ohc.empty: return
#     set_font()
#     df = df_ohc.sort_values("cohesive_ratio", ascending=False).copy()
#     fig, ax = plt.subplots(figsize=(8,4), dpi=DPI)
#     x = np.arange(len(df))
#     ax.bar(x, df["cohesive_ratio"].values, color="#59A14F", alpha=0.9)
#     ax.set_xticks(x); ax.set_xticklabels([f"OH {int(c)}" for c in df["OH_cluster"]], rotation=35, ha="right")
#     ax.set_ylim(0,1.05); ax.set_ylabel("Cohesive ratio (pair-level)")
#     title = f"Fig 4.2.4.4: OH-cluster cohesion in FP space ({mode} × {index})"
#     if main_text: title += " [Main]"
#     ax.set_title(title)
#     for xi, v in zip(x, df["cohesive_ratio"].values): ax.text(xi, v+0.02, f"{v:.2f}", ha="center", va="bottom", fontsize=8)
#     ax.grid(axis="y", linestyle="--", linewidth=0.4, alpha=0.6)
#     fig.tight_layout()
#     fname = f"Fig_4.2.4.4_OHcluster_cohesion_{mode}_{index}{'_MAIN' if main_text else ''}"
#     fig.savefig(outdir / f"{fname}.png", bbox_inches="tight"); fig.savefig(outdir / f"{fname}.pdf", bbox_inches="tight")
#     plt.close(fig)

# def plot_fpcluster_consistency(df_fpc: pd.DataFrame, outdir: Path, mode: str, index: str, main_text:bool=False):
#     if df_fpc is None or df_fpc.empty: return
#     set_font()
#     df = df_fpc.sort_values("cohesive_ratio", ascending=False).copy()
#     fig, ax = plt.subplots(figsize=(8,4), dpi=DPI)
#     x = np.arange(len(df))
#     ax.bar(x, df["cohesive_ratio"].values, color="#59A14F", alpha=0.9)
#     ax.set_xticks(x); ax.set_xticklabels([f"FP {int(c)}" for c in df["FP_cluster"]], rotation=35, ha="right")
#     ax.set_ylim(0,1.05); ax.set_ylabel("Cohesive ratio (pair-level in OH space)")
#     title = f"Fig 4.2.4.4R: FP-cluster cohesion in OH space ({mode} × {index})"
#     if main_text: title += " [Main]"
#     ax.set_title(title)
#     for xi, v in zip(x, df["cohesive_ratio"].values): ax.text(xi, v+0.02, f"{v:.2f}", ha="center", va="bottom", fontsize=8)
#     ax.grid(axis="y", linestyle="--", linewidth=0.4, alpha=0.6)
#     fig.tight_layout()
#     fname = f"Fig_4.2.4.4R_FPcluster_cohesion_{mode}_{index}{'_MAIN' if main_text else ''}"
#     fig.savefig(outdir / f"{fname}.png", bbox_inches="tight"); fig.savefig(outdir / f"{fname}.pdf", bbox_inches="tight")
#     plt.close(fig)

# def plot_mini_sankey_material_to_fp(df_mat: pd.DataFrame, outdir: Path, mode: str, index: str, topN=20, main_text:bool=False):
#     if df_mat is None or df_mat.empty: return
#     set_font()
#     df = df_mat.sort_values("FP_purity", ascending=False).head(topN).copy()
#     if df.empty: return
#     widths = (df["FP_purity"] + 0.2) / (df["FP_purity"].max() + 0.2)
#     y = 0.0
#     fig, ax = plt.subplots(figsize=(10, 0.35*len(df)+2), dpi=DPI)
#     for i, r in df.iterrows():
#         w = widths.loc[i]
#         ax.plot([0, 1], [y, y], lw=8*w, alpha=0.9, color="#4E79A7")
#         ax.text(-0.02, y, str(r["material"]), ha="right", va="center", fontsize=8)
#         fp_major = r.get("FP_major_cluster"); right_lbl = f"FP {int(fp_major)}" if pd.notna(fp_major) else "FP –"
#         ax.text(1.02, y, right_lbl, ha="left", va="center", fontsize=8, color="#555")
#         y += 0.5
#     ax.set_xlim(-0.25, 1.25); ax.set_ylim(-0.5, y-0.0); ax.axis("off")
#     title = f"Fig 4.2.4.4: Material → FP major cluster (top {topN}) ({mode} × {index})"
#     if main_text: title += " [Main]"
#     ax.set_title(title, loc="left")
#     fig.tight_layout()
#     fname = f"Fig_4.2.4.4_material-to-FPmajor_miniAlluvial_{mode}_{index}{'_MAIN' if main_text else ''}"
#     fig.savefig(outdir / f"{fname}.png", bbox_inches="tight"); fig.savefig(outdir / f"{fname}.pdf", bbox_inches="tight")
#     plt.close(fig)

# def plot_mini_sankey_fragment_to_oh(df_frag: pd.DataFrame, outdir: Path, mode: str, index: str, topN=20, main_text:bool=False):
#     if df_frag is None or df_frag.empty: return
#     set_font()
#     df = df_frag.sort_values("OH_purity", ascending=False).head(topN).copy()
#     if df.empty: return
#     widths = (df["OH_purity"] + 0.2) / (df["OH_purity"].max() + 0.2)
#     y = 0.0
#     fig, ax = plt.subplots(figsize=(10, 0.35*len(df)+2), dpi=DPI)
#     for i, r in df.iterrows():
#         w = widths.loc[i]
#         ax.plot([0, 1], [y, y], lw=8*w, alpha=0.9, color="#E15759")
#         ax.text(-0.02, y, str(r["fragment"]), ha="right", va="center", fontsize=8)
#         oh_major = r.get("OH_major_cluster"); right_lbl = f"OH {int(oh_major)}" if pd.notna(oh_major) else "OH –"
#         ax.text(1.02, y, right_lbl, ha="left", va="center", fontsize=8, color="#555")
#         y += 0.5
#     ax.set_xlim(-0.25, 1.25); ax.set_ylim(-0.5, y-0.0); ax.axis("off")
#     title = f"Fig 4.2.4.4R: Fragment → OH major cluster (top {topN}) ({mode} × {index})"
#     if main_text: title += " [Main]"
#     ax.set_title(title, loc="left")
#     fig.tight_layout()
#     fname = f"Fig_4.2.4.4R_fragment-to-OHmajor_miniAlluvial_{mode}_{index}{'_MAIN' if main_text else ''}"
#     fig.savefig(outdir / f"{fname}.png", bbox_inches="tight"); fig.savefig(outdir / f"{fname}.pdf", bbox_inches="tight")
#     plt.close(fig)

# # ========= OH→FP（前方向） =========
# def build_forward_direction():
#     set_font()
#     Xoh0 = load_features_bin(FEATURES_OH)
#     Xfp0 = load_features_bin(FEATURES_FP)
#     common_samples = Xoh0.index.intersection(Xfp0.index)
#     if len(common_samples)==0:
#         raise RuntimeError("No common samples between features_OH_onlyMat and features_FP_onlyMat.")
#     Xoh0 = Xoh0.loc[common_samples]; Xfp0 = Xfp0.loc[common_samples]

#     oh_latest = latest_by_mode_index(FIGS_OH, "OH")
#     fp_latest = latest_by_mode_index(FIGS_FP, "FP")
#     keys = [(m,i) for m in DIMS for i in INDICES if (m,i) in oh_latest and (m,i) in fp_latest]
#     if not keys:
#         raise RuntimeError("OH/FP の ClusterAssign に共通 (mode,index) が見つかりません。")

#     summary_rows = []

#     for (mode, index) in keys:
#         print(f"[FWD] {mode} × {index}")
#         oh_assign = read_var_cluster(oh_latest[(mode,index)])
#         fp_assign = read_var_cluster(fp_latest[(mode,index)])

#         # 材料 → FP 主分布
#         materials = [m for m in oh_assign.index if m in Xoh0.columns]
#         rows_mat = []
#         for mat in materials:
#             n_mat = int(Xoh0[mat].sum())
#             if n_mat <= 0: continue
#             maj, purity, n_s, dist = material_to_fp_distribution(mat, Xoh0, Xfp0, fp_assign, threshold=FRAG_PREV_THRESHOLD)
#             oh_c = oh_assign.get(mat, np.nan)
#             H = entropy_from_dist(dist); k_eff = len(dist) if dist else 0
#             rows_mat.append({
#                 "mode": mode, "index": index,
#                 "material": mat,
#                 "OH_cluster": int(oh_c) if pd.notna(oh_c) else np.nan,
#                 "n_samples_with_material": n_s,
#                 "FP_major_cluster": maj,
#                 "FP_purity": purity,
#                 "FP_entropy": H,
#                 "FP_k_effective": k_eff,
#                 "FP_dist": ";".join(f"{k}:{v:.3f}" for k,v in sorted(dist.items())) if dist else ""
#             })
#         df_mat = pd.DataFrame(rows_mat)
#         out_csv1 = OUT_VARMAP / f"Table_4.2.4.4_material-to-FPmajor_{mode}_{index}.csv"
#         if len(df_mat): df_mat.to_csv(out_csv1, index=False, encoding="utf-8-sig")

#         # 材料ペア（同一OHクラスタ）で一貫性
#         rows_pair = []
#         if len(df_mat):
#             def parse_dist(s):
#                 if isinstance(s, str) and s:
#                     d={}; 
#                     for t in s.split(";"):
#                         k,v = t.split(":"); d[int(k)] = float(v)
#                     return d
#                 return {}
#             dists = {r.material: parse_dist(r.FP_dist) for r in df_mat.itertuples()}
#             all_fp_clusters = sorted({k for d in dists.values() for k in d.keys()})
#             def vec_of(dist):
#                 return np.array([dist.get(k, 0.0) for k in all_fp_clusters], dtype=float)

#             for k, grp in df_mat.dropna(subset=["OH_cluster"]).groupby("OH_cluster"):
#                 mats = grp["material"].tolist()
#                 for a, b in combinations(mats, 2):
#                     da, db = dists.get(a, {}), dists.get(b, {})
#                     va, vb = vec_of(da), vec_of(db)
#                     if va.sum()==0 and vb.sum()==0:
#                         cos_sim = np.nan; jsd = np.nan
#                     else:
#                         cos_sim = 1.0 - (cosine(va, vb) if (va.sum()>0 and vb.sum()>0) else 1.0)
#                         jsd = js_divergence(va, vb) if (va.sum()>0 and vb.sum()>0) else np.nan
#                     maj_same = (df_mat.loc[df_mat["material"]==a, "FP_major_cluster"].values[0] ==
#                                 df_mat.loc[df_mat["material"]==b, "FP_major_cluster"].values[0])
#                     purity_min = float(min(
#                         df_mat.loc[df_mat["material"]==a, "FP_purity"].values[0] if len(df_mat.loc[df_mat["material"]==a, "FP_purity"]) else 0.0,
#                         df_mat.loc[df_mat["material"]==b, "FP_purity"].values[0] if len(df_mat.loc[df_mat["material"]==b, "FP_purity"]) else 0.0
#                     ))
#                     rows_pair.append({
#                         "mode": mode, "index": index,
#                         "OH_cluster": int(k),
#                         "material_A": a, "material_B": b,
#                         "FP_major_same": bool(maj_same),
#                         "FP_purity_min": purity_min,
#                         "FP_cosine_similarity": cos_sim,
#                         "FP_JS_divergence": jsd
#                     })
#         df_pair = pd.DataFrame(rows_pair)
#         out_csv2 = OUT_VARMAP / f"Table_4.2.4.4_material-pair_cohesion_{mode}_{index}.csv"
#         if len(df_pair): df_pair.to_csv(out_csv2, index=False, encoding="utf-8-sig")

#         # OHクラスタ一貫性
#         if len(df_pair):
#             def label_cohesion(r, purity_thr=PURITY_FLAG_THR, cos_thr=COS_SIM_THR, js_thr=JS_DIV_THR):
#                 return bool(
#                     (r["FP_major_same"] and r["FP_purity_min"] >= purity_thr) or
#                     (r["FP_cosine_similarity"] >= cos_thr) or
#                     (pd.notna(r["FP_JS_divergence"]) and r["FP_JS_divergence"] <= js_thr)
#                 )
#             df_pair["cohesive_flag"] = df_pair.apply(label_cohesion, axis=1)
#             df_ohc = (df_pair.groupby(["mode","index","OH_cluster"], as_index=False)
#                              .agg(n_pairs=("cohesive_flag","size"),
#                                   n_cohesive=("cohesive_flag","sum")))
#             df_ohc["cohesive_ratio"] = df_ohc["n_cohesive"] / df_ohc["n_pairs"].replace(0, np.nan)
#         else:
#             df_ohc = pd.DataFrame()

#         out_csv3 = OUT_VARMAP / f"Table_4.2.4.4_OHcluster_cohesion_{mode}_{index}.csv"
#         if len(df_ohc): df_ohc.to_csv(out_csv3, index=False, encoding="utf-8-sig")

#         # 図（本文/付録の振分）
#         is_main = (mode, index) in MAIN_PAIRS
#         out_dir = OUT_MAIN / f"FigSet_4.2.4.4_{mode}_{index}" if is_main else OUT_APPENDIX / f"FigSet_4.2.4.4_{mode}_{index}"
#         out_dir.mkdir(parents=True, exist_ok=True)
#         plot_material_scatter(df_mat, out_dir, mode, index, top_labels=15, main_text=is_main)
#         plot_pair_distributions_forward(df_pair, out_dir, mode, index, main_text=is_main)
#         plot_ohcluster_consistency(df_ohc, out_dir, mode, index, main_text=is_main)
#         plot_mini_sankey_material_to_fp(df_mat, out_dir, mode, index, topN=20, main_text=is_main)

#         # サマリー
#         summary_rows.append({
#             "mode": mode, "index": index,
#             "n_materials": len(df_mat),
#             "mean_FP_purity": df_mat["FP_purity"].dropna().mean() if "FP_purity" in df_mat else np.nan,
#             "mean_FP_entropy": df_mat["FP_entropy"].dropna().mean() if "FP_entropy" in df_mat else np.nan,
#             "pair_FP_major_same_rate": df_pair["FP_major_same"].mean() if len(df_pair) else np.nan,
#             "pair_mean_cosine_similarity": df_pair["FP_cosine_similarity"].dropna().mean() if len(df_pair) else np.nan,
#             "pair_mean_JS_divergence": df_pair["FP_JS_divergence"].dropna().mean() if len(df_pair) else np.nan,
#             "OHcluster_median_cohesive_ratio": df_ohc["cohesive_ratio"].median() if len(df_ohc) else np.nan
#         })

#     # 全条件サマリー（前方向）
#     if summary_rows:
#         df_sum = pd.DataFrame(summary_rows).sort_values(["mode","index"])
#         outsum_all = OUT_OHFP_ROOT / "Table_4.2.4.4_summary_all_conditions.csv"
#         df_sum.to_csv(outsum_all, index=False, encoding="utf-8-sig")
#         # 本文向け（cumeig のみ）
#         df_main = df_sum[df_sum["mode"]=="cumeig"].copy()
#         outsum_main = OUT_MAIN / "Table_4.2.4.4_summary_cumeig_only.csv"
#         df_main.to_csv(outsum_main, index=False, encoding="utf-8-sig")

# # ========= FP→OH（逆方向） =========
# def build_reverse_direction():
#     set_font()
#     Xoh0 = load_features_bin(FEATURES_OH)
#     Xfp0 = load_features_bin(FEATURES_FP)
#     common = Xoh0.index.intersection(Xfp0.index)
#     if len(common)==0:
#         raise RuntimeError("No common samples between features_OH_onlyMat and features_FP_onlyMat.")
#     Xoh0 = Xoh0.loc[common]; Xfp0 = Xfp0.loc[common]

#     oh_latest = latest_by_mode_index(FIGS_OH, "OH")
#     fp_latest = latest_by_mode_index(FIGS_FP, "FP")
#     keys = [(m,i) for m in DIMS for i in INDICES if (m,i) in oh_latest and (m,i) in fp_latest]
#     if not keys:
#         raise RuntimeError("No common (mode,index) between figs_OH and figs_FP.")

#     summary_rows = []

#     for (mode,index) in keys:
#         print(f"[REV] {mode} × {index}")
#         oh_assign = read_var_cluster(oh_latest[(mode,index)])
#         fp_assign = read_var_cluster(fp_latest[(mode,index)])

#         # フラグメント → OH 主分布
#         fragments = [f for f in fp_assign.index if f in Xfp0.columns]
#         rows_frag=[]
#         for frag in fragments:
#             n_frag = int(Xfp0[frag].sum())
#             if n_frag<=0: continue
#             maj, purity, n_s, dist = fragment_to_oh_distribution(frag, Xfp0, Xoh0, oh_assign, threshold=MAT_PREV_THRESHOLD)
#             fp_c = fp_assign.get(frag, np.nan)
#             H = entropy_from_dist(dist); k_eff = len(dist) if dist else 0
#             rows_frag.append({
#                 "mode":mode,"index":index,
#                 "fragment":frag,
#                 "FP_cluster": int(fp_c) if pd.notna(fp_c) else np.nan,
#                 "n_samples_with_fragment": n_s,
#                 "OH_major_cluster": maj,
#                 "OH_purity": purity,
#                 "OH_entropy": H,
#                 "OH_k_effective": k_eff,
#                 "OH_dist": ";".join(f"{k}:{v:.3f}" for k,v in sorted(dist.items())) if dist else ""
#             })
#         df_frag = pd.DataFrame(rows_frag)
#         out_csv1 = OUT_R_CSV / f"Table_4.2.4.4R_fragment-to-OHmajor_{mode}_{index}.csv"
#         if len(df_frag): df_frag.to_csv(out_csv1, index=False, encoding="utf-8-sig")

#         # 同一FPクラスタ内ペアの材料側一貫性
#         rows_pair=[]
#         if len(df_frag):
#             def parse_dist(s):
#                 if isinstance(s,str) and s:
#                     d={}
#                     for t in s.split(";"):
#                         k,v = t.split(":"); d[int(k)] = float(v)
#                     return d
#                 return {}
#             dists = {r.fragment: parse_dist(r.OH_dist) for r in df_frag.itertuples()}
#             all_oh_clusters = sorted({k for d in dists.values() for k in d.keys()})
#             def vec_of(dist):
#                 return np.array([dist.get(k, 0.0) for k in all_oh_clusters], dtype=float)

#             for k, grp in df_frag.dropna(subset=["FP_cluster"]).groupby("FP_cluster"):
#                 frags = grp["fragment"].tolist()
#                 for a,b in combinations(frags,2):
#                     da, db = dists.get(a,{}), dists.get(b,{})
#                     va, vb = vec_of(da), vec_of(db)
#                     if va.sum()==0 and vb.sum()==0:
#                         cos_sim=np.nan; jsd=np.nan
#                     else:
#                         cos_sim = 1.0 - (cosine(va, vb) if (va.sum()>0 and vb.sum()>0) else 1.0)
#                         jsd = js_divergence(va, vb) if (va.sum()>0 and vb.sum()>0) else np.nan
#                     maj_same = (df_frag.loc[df_frag["fragment"]==a,"OH_major_cluster"].values[0] ==
#                                 df_frag.loc[df_frag["fragment"]==b,"OH_major_cluster"].values[0])
#                     purity_min = float(min(
#                         df_frag.loc[df_frag["fragment"]==a,"OH_purity"].values[0] if len(df_frag.loc[df_frag["fragment"]==a,"OH_purity"]) else 0.0,
#                         df_frag.loc[df_frag["fragment"]==b,"OH_purity"].values[0] if len(df_frag.loc[df_frag["fragment"]==b,"OH_purity"]) else 0.0
#                     ))
#                     rows_pair.append({
#                         "mode":mode,"index":index,
#                         "FP_cluster":int(k),
#                         "fragment_A":a,"fragment_B":b,
#                         "OH_major_same":bool(maj_same),
#                         "OH_purity_min": purity_min,
#                         "OH_cosine_similarity": cos_sim,
#                         "OH_JS_divergence": jsd
#                     })
#         df_pair = pd.DataFrame(rows_pair)
#         out_csv2 = OUT_R_CSV / f"Table_4.2.4.4R_fragment-pair_cohesion_{mode}_{index}.csv"
#         if len(df_pair): df_pair.to_csv(out_csv2, index=False, encoding="utf-8-sig")

#         if len(df_pair):
#             def label_cohesion(r, purity_thr=PURITY_FLAG_THR, cos_thr=COS_SIM_THR, js_thr=JS_DIV_THR):
#                 return bool(
#                     (r["OH_major_same"] and r["OH_purity_min"]>=purity_thr) or
#                     (pd.notna(r["OH_cosine_similarity"]) and r["OH_cosine_similarity"]>=cos_thr) or
#                     (pd.notna(r["OH_JS_divergence"]) and r["OH_JS_divergence"]<=js_thr)
#                 )
#             df_pair["cohesive_flag"] = df_pair.apply(label_cohesion, axis=1)
#             df_fpc = (df_pair.groupby(["mode","index","FP_cluster"], as_index=False)
#                              .agg(n_pairs=("cohesive_flag","size"),
#                                   n_cohesive=("cohesive_flag","sum")))
#             df_fpc["cohesive_ratio"] = df_fpc["n_cohesive"]/df_fpc["n_pairs"].replace(0, np.nan)
#         else:
#             df_fpc = pd.DataFrame()
#         out_csv3 = OUT_R_CSV / f"Table_4.2.4.4R_FPcluster_cohesion_{mode}_{index}.csv"
#         if len(df_fpc): df_fpc.to_csv(out_csv3, index=False, encoding="utf-8-sig")

#         # 図（本文/付録）
#         is_main = (mode,index) in MAIN_PAIRS
#         out_dir = (OUT_R_MAIN if is_main else OUT_R_APPENDIX) / f"FigSet_4.2.4.4R_{mode}_{index}"
#         out_dir.mkdir(parents=True, exist_ok=True)
#         plot_fragment_scatter(df_frag, out_dir, mode, index, top_labels=15, main_text=is_main)
#         plot_pair_distributions_reverse(df_pair, out_dir, mode, index, main_text=is_main)
#         plot_fpcluster_consistency(df_fpc, out_dir, mode, index, main_text=is_main)
#         plot_mini_sankey_fragment_to_oh(df_frag, out_dir, mode, index, topN=20, main_text=is_main)

#         # サマリー
#         summary_rows.append({
#             "mode":mode,"index":index,
#             "n_fragments": len(df_frag),
#             "mean_OH_purity": df_frag["OH_purity"].dropna().mean() if "OH_purity" in df_frag else np.nan,
#             "mean_OH_entropy": df_frag["OH_entropy"].dropna().mean() if "OH_entropy" in df_frag else np.nan,
#             "pair_OH_major_same_rate": df_pair["OH_major_same"].mean() if len(df_pair) else np.nan,
#             "pair_mean_cosine_similarity": df_pair["OH_cosine_similarity"].dropna().mean() if len(df_pair) else np.nan,
#             "pair_mean_JS_divergence": df_pair["OH_JS_divergence"].dropna().mean() if len(df_pair) else np.nan,
#             "FPcluster_median_cohesive_ratio": df_fpc["cohesive_ratio"].median() if len(df_fpc) else np.nan
#         })

#     if summary_rows:
#         df_sum = pd.DataFrame(summary_rows).sort_values(["mode","index"])
#         outsum = OUT_R_ROOT / "Table_4.2.4.4R_summary_all_conditions.csv"
#         df_sum.to_csv(outsum, index=False, encoding="utf-8-sig")

# # ========= signless vs new: OH→FP 側の一致度（OH/FP個別） =========
# def collect_assign_labels(figs_dir: Path, ds: str) -> pd.DataFrame | None:
#     pat = re.compile(rf"^ClusterAssign_(top3|cumeig)_(silhouette|dunn|gap|ch|db|ptbiserial)_{ds}_.*\.csv$")
#     files = [p for p in figs_dir.glob("ClusterAssign_*.csv") if pat.match(p.name)]
#     parts=[]
#     for f in files:
#         m = re.match(r"ClusterAssign_(top3|cumeig)_([A-Za-z]+)_", f.name)
#         if not m: continue
#         mode, index = m.groups()
#         df = read_csv_quiet(f)
#         if df is None: continue
#         if {"Variable","Cluster"}.issubset(df.columns):
#             ids = df["Variable"].astype(str).values
#             cl  = df["Cluster"].values
#         elif df.shape[1]==1:
#             ids = df.index.astype(str).values
#             cl  = df.iloc[:,0].values
#         else:
#             continue
#         codes = pd.Categorical(cl).codes
#         codes = pd.Series(codes).replace({-1: np.nan}).values
#         parts.append(pd.DataFrame({"id": ids, f"{mode}_{index}": codes}))
#     if not parts: return None
#     out = parts[0]
#     for p in parts[1:]:
#         out = out.merge(p, on="id", how="outer")
#     return out.drop_duplicates(subset=["id"])

# def load_signless_labels(signless_dir: Path) -> pd.DataFrame | None:
#     cands = sorted(signless_dir.glob("cluster_labels_*.csv"), key=os.path.getmtime, reverse=True)
#     if not cands: return None
#     df = read_csv_quiet(cands[0])
#     if df is None or "id" not in df.columns: return None
#     keep = [c for c in df.columns if str(c).startswith("linear_")]
#     if not keep: return None
#     out = df[["id"]+keep].copy()
#     out.columns = ["id"] + [c.replace("linear_","",1) for c in keep]
#     return out

# def pairwise_scores(dfA: pd.DataFrame|None, dfB: pd.DataFrame|None) -> pd.DataFrame:
#     cols = ["condition","n","kA","kB","ARI","NMI","AMI"]
#     if dfA is None or dfB is None: return pd.DataFrame(columns=cols)
#     colsA = [c for c in dfA.columns if c!="id"]
#     colsB = [c for c in dfB.columns if c!="id"]
#     common = sorted(set(colsA).intersection(colsB))
#     if not common: return pd.DataFrame(columns=cols)
#     merged = dfA.merge(dfB, on="id", suffixes=(".A",".B"))
#     rows=[]
#     for cn in common:
#         a = merged[f"{cn}.A"].values
#         b = merged[f"{cn}.B"].values
#         mask = ~pd.isna(a) & ~pd.isna(b)
#         if mask.sum() < 2: continue
#         a = a[mask].astype(int); b = b[mask].astype(int)
#         rows.append([
#             cn, int(mask.sum()),
#             int(len(np.unique(a))), int(len(np.unique(b))),
#             float(adjusted_rand_score(a,b)),
#             float(normalized_mutual_info_score(a,b)),
#             float(adjusted_mutual_info_score(a,b)),
#         ])
#     return pd.DataFrame(rows, columns=cols)

# def lighten(color, factor=0.5):
#     r,g,b = matplotlib.colors.to_rgb(color)
#     return (1 - factor) + factor*r, (1 - factor) + factor*g, (1 - factor) + factor*b

# def save_bar_contrast_with_k(df_long: pd.DataFrame, T_meta: pd.DataFrame, out_dir: Path, title: str, fname_core: str, fade_outside: bool):
#     set_font()
#     INDEX_LIST = ["silhouette","dunn","gap","ch","db","ptbiserial"]
#     DATASETS = ["OH","FP"]

#     df = df_long.copy()
#     df["mode"]  = pd.Categorical(df["mode"], categories=["top3","cumeig"], ordered=True)
#     df["index"] = pd.Categorical(df["index"], categories=INDEX_LIST, ordered=True)
#     df = df.sort_values(["index","mode","Dataset"])

#     # x slots
#     x_labels = [f"{m}\n{ix}" for m in ["top3","cumeig"] for ix in INDEX_LIST]
#     grid=[]
#     for m in ["top3","cumeig"]:
#         for ix in INDEX_LIST:
#             sl = df[(df["mode"]==m) & (df["index"]==ix)]
#             grid.append(sl.set_index("Dataset")["ARI"].reindex(DATASETS))
#     X = pd.concat(grid, axis=1).T
#     X.index = x_labels

#     # META lookup
#     M = T_meta.copy()
#     M["row_key"] = M["mode"] + "\n" + M["index"]
#     M["is_target"] = (M[["kA","kB"]].max(axis=1) <= K_MAX_KEEP)
#     META = (M.set_index(["row_key","Dataset"])[["kA","kB","is_target"]]
#               .reindex(pd.MultiIndex.from_product([x_labels, DATASETS], names=["row_key","Dataset"])))

#     color_OH = "#1f77b4"; color_FP="#ff7f0e"
#     fade_OH  = lighten(color_OH, 0.7); fade_FP = lighten(color_FP, 0.7)

#     fig, ax = plt.subplots(figsize=(12,6), dpi=DPI)
#     width = 0.38; x = np.arange(X.shape[0])

#     for i, ds in enumerate(DATASETS):
#         base_color = color_OH if ds=="OH" else color_FP
#         base_fade  = fade_OH  if ds=="OH" else fade_FP
#         xs = x + (i-0.5)*width
#         vals = X[ds].values
#         for xi, val, row_key in zip(xs, vals, X.index):
#             if (row_key, ds) in META.index:
#                 kA = META.loc[(row_key, ds), "kA"]; kB = META.loc[(row_key, ds), "kB"]
#                 is_t = bool(META.loc[(row_key, ds), "is_target"])
#             else:
#                 kA = np.nan; kB = np.nan; is_t=False
#             fc = base_color if (not fade_outside or is_t) else base_fade
#             alpha = 0.95 if (not fade_outside or is_t) else 0.5
#             ax.bar(xi, val, width, color=fc, alpha=alpha, edgecolor="none")
#             if not np.isnan(val):
#                 label = f"{val:.2f}\n({int(kA) if not np.isnan(kA) else '-'}" \
#                         f"/{int(kB) if not np.isnan(kB) else '-'})"
#                 ax.text(xi, val + 0.015, label, ha="center", va="bottom", fontsize=8)

#     ax.set_xticks(x); ax.set_xticklabels(X.index, rotation=35, ha="right")
#     ax.set_ylim(0,1.05); ax.set_ylabel("ARI")
#     ax.set_title(title)
#     legend_elems = [Patch(facecolor=color_OH, edgecolor="none", alpha=0.95, label="OH (k≤30)"),
#                     Patch(facecolor=color_FP, edgecolor="none", alpha=0.95, label="FP (k≤30)")]
#     if fade_outside:
#         legend_elems += [Patch(facecolor=fade_OH, edgecolor="none", alpha=0.50, label="OH (k>30)"),
#                          Patch(facecolor=fade_FP, edgecolor="none", alpha=0.50, label="FP (k>30)")]
#     ax.legend(handles=legend_elems, loc="upper left", bbox_to_anchor=(1.02, 1.0))
#     ax.grid(axis="y", linestyle="--", linewidth=0.4, alpha=0.6)
#     fig.tight_layout()
#     out_dir.mkdir(parents=True, exist_ok=True)
#     fig.savefig(out_dir / f"{fname_core}.png", bbox_inches="tight")
#     fig.savefig(out_dir / f"{fname_core}.pdf", bbox_inches="tight")
#     plt.close(fig)

# def build_ohfp_contrast():
#     # signless vs new を OH/FP で独立に算出し、本文（k≤30）/付録（allK）を出力
#     all_tbl_main=[]; all_for_contrast_main=[]; all_tbl_allK=[]; all_for_contrast_allK=[]
#     for ds, figs in [("OH",FIGS_OH), ("FP",FIGS_FP)]:
#         A = collect_assign_labels(figs, ds)
#         S = load_signless_labels(SIGNLESS_DIR[ds])
#         T = pairwise_scores(A, S)
#         if T.empty:
#             print(f"[WARN] {ds}: no common conditions")
#             continue
#         T["mode"]    = T["condition"].str.replace(r"_(.*)$", "", regex=True)
#         T["index"]   = T["condition"].str.replace(r"^(.*)_", "", regex=True)
#         T["Dataset"] = ds
#         T["k_max"]   = T[["kA","kB"]].max(axis=1)

#         all_tbl_allK.append(T[["condition","Dataset","n","kA","kB","ARI","NMI","AMI","mode","index","k_max"]])
#         all_for_contrast_allK.append(T[["Dataset","mode","index","ARI","kA","kB","k_max"]])

#         T30 = T[(T["k_max"].isna()) | (T["k_max"]<=K_MAX_KEEP)].copy()
#         if len(T30):
#             all_tbl_main.append(T30[["condition","Dataset","n","kA","kB","ARI","NMI","AMI","mode","index","k_max"]])
#             all_for_contrast_main.append(T30[["Dataset","mode","index","ARI","kA","kB","k_max"]])

#     if all_tbl_main:
#         T_all = pd.concat(all_tbl_main, ignore_index=True)
#         out_csv = OUT_MAIN / "Table_4.2.4.4_OHFP-contrast_summary_k-le30.csv"
#         T_all.sort_values(["Dataset","mode","index"]).to_csv(out_csv, index=False, encoding="utf-8-sig")
#         print(f"[SAVE] {out_csv}")

#         contrast_src = pd.concat(all_for_contrast_main, ignore_index=True)
#         for mode,index in MAIN_PAIRS:
#             sub = contrast_src[(contrast_src["mode"]==mode)&(contrast_src["index"]==index)]
#             out_dir = OUT_MAIN / f"FigSet_4.2.4.4_{mode}_{index}"
#             save_bar_contrast_with_k(
#                 df_long=sub,
#                 T_meta=T_all[(T_all["mode"]==mode) & (T_all["index"]==index)][["Dataset","mode","index","kA","kB","k_max"]],
#                 out_dir=out_dir,
#                 title=f"Fig 4.2.4.4: OH vs FP (ARI of new vs signless, {mode}×{index}, k≤30)",
#                 fname_core="Fig_4.2.4.4_OHFP-contrast_ARI-bars",
#                 fade_outside=False
#             )
#     else:
#         print("[MAIN] No k≤30 rows for OH→FP contrast.")

#     if all_tbl_allK:
#         T_allK = pd.concat(all_tbl_allK, ignore_index=True)
#         out_csv_apx = OUT_APPENDIX / "Appendix_Table_allK_OHFP-contrast_summary.csv"
#         T_allK.sort_values(["Dataset","mode","index"]).to_csv(out_csv_apx, index=False, encoding="utf-8-sig")
#         print(f"[SAVE] {out_csv_apx}")

#         contrast_allK = pd.concat(all_for_contrast_allK, ignore_index=True)
#         save_bar_contrast_with_k(
#             df_long=contrast_allK[["Dataset","mode","index","ARI"]],
#             T_meta=T_allK[["Dataset","mode","index","kA","kB","k_max"]],
#             out_dir=OUT_APPENDIX,
#             title="Appendix: OH vs FP (ARI of new vs signless, all k)",
#             fname_core="Appendix_Fig_allK_OHFP-contrast_ARI-bars",
#             fade_outside=True
#         )

# # ========= 双方向まとめ =========
# def build_bidirectional_summary():
#     fwd = OUT_OHFP_ROOT / "Table_4.2.4.4_summary_all_conditions.csv"
#     rev = OUT_R_ROOT    / "Table_4.2.4.4R_summary_all_conditions.csv"
#     if not fwd.exists() or not rev.exists():
#         print("[INFO] Skip bidirectional summary (source tables not found).")
#         return
#     A = pd.read_csv(fwd); R = pd.read_csv(rev)

#     # 前方向：各 (mode,index) の代表スコア（ここでは mean FP_purity などではなく signless ではないため簡略化）
#     # 便宜上、前方向の“整合度スコア”として pair_FP_major_same_rate を採用（必要に応じて他の合成指標に変更可）
#     fwd_sc = (A.groupby(["mode","index"])["pair_FP_major_same_rate"].mean().rename("forward_score")).reset_index()

#     # 逆方向：複合スコア（正の指標は↑、負の指標は↓）
#     def minmax(s):
#         s2 = s.copy()
#         if s2.isna().all(): return s2.fillna(0.5)
#         s2 = s2.fillna(s2.mean()); mn, mx = s2.min(), s2.max()
#         if np.isclose(mx, mn): return pd.Series(0.5, index=s2.index)
#         return (s2 - mn) / (mx - mn)

#     rev2 = R.copy()
#     pos = [c for c in ["mean_OH_purity","pair_OH_major_same_rate","pair_mean_cosine_similarity","FPcluster_median_cohesive_ratio"] if c in rev2.columns]
#     neg = [c for c in ["mean_OH_entropy","pair_mean_JS_divergence"] if c in rev2.columns]
#     parts = []
#     for c in pos: parts.append(minmax(rev2[c]))
#     for c in neg: parts.append(1 - minmax(rev2[c]))
#     if parts:
#         rev2["reverse_score"] = pd.concat(parts, axis=1).mean(axis=1)
#     else:
#         rev2["reverse_score"] = np.nan
#     rev_sc = rev2.groupby(["mode","index"])["reverse_score"].mean().reset_index()

#     M = fwd_sc.merge(rev_sc, on=["mode","index"], how="outer").sort_values(["mode","index"])
#     out_csv = OUT_BIDIR / "Table_4.2.4.4_bidirectional_summary.csv"
#     M.to_csv(out_csv, index=False, encoding="utf-8-sig")
#     print(f"[SAVE] {out_csv}")

#     # 図
#     set_font()
#     fig, ax = plt.subplots(figsize=(10,5), dpi=DPI)
#     lbls = [f"{m}-{i}" for m,i in zip(M["mode"], M["index"])]
#     x = np.arange(len(M)); w = 0.38
#     ax.bar(x-w/2, M["forward_score"].values, width=w, label="OH→FP (proxy score)", alpha=0.9)
#     ax.bar(x+w/2, M["reverse_score"].values, width=w, label="FP→OH (composite)", alpha=0.9)
#     ax.set_xticks(x); ax.set_xticklabels(lbls, rotation=35, ha="right")
#     ax.set_ylim(0,1.05); ax.set_ylabel("Score (0–1)")
#     ax.set_title("Fig 4.2.4.4: Bidirectional alignment summary")
#     ax.legend(); ax.grid(axis="y", linestyle="--", linewidth=0.4, alpha=0.6)
#     fig.tight_layout()
#     fig.savefig(OUT_BIDIR / "Fig_4.2.4.4_bidirectional_comparison.png", bbox_inches="tight")
#     fig.savefig(OUT_BIDIR / "Fig_4.2.4.4_bidirectional_comparison.pdf", bbox_inches="tight")
#     plt.close(fig)

# # ========= （任意）k固定 Permutation / Bootstrap 検証 =========
# def permutation_pvalue(yA: np.ndarray, yB: np.ndarray, n_perm=N_PERM):
#     obs = adjusted_rand_score(yA, yB)
#     null = np.zeros(n_perm, dtype=float)
#     for i in range(n_perm):
#         perm = rng.permutation(yB)
#         null[i] = adjusted_rand_score(yA, perm)
#     p = float((np.sum(null >= obs) + 1) / (n_perm + 1))  # 右片側
#     return obs, null, p

# def bootstrap_ci(yA: np.ndarray, yB: np.ndarray, n_boot=N_BOOT, alpha=0.05):
#     n = len(yA)
#     vals = np.zeros(n_boot, dtype=float)
#     for i in range(n_boot):
#         idx = rng.integers(0, n, size=n)
#         vals[i] = adjusted_rand_score(yA[idx], yB[idx])
#     lo = float(np.quantile(vals, alpha/2)); hi = float(np.quantile(vals, 1-alpha/2))
#     return (lo, hi), vals

# def plot_perm_hist(null_vals: np.ndarray, obs_ari: float, mode: str, index: str, outdir: Path):
#     set_font()
#     fig, ax = plt.subplots(figsize=(7,4), dpi=DPI)
#     ax.hist(null_vals, bins=30, alpha=0.9)
#     ax.axvline(obs_ari, color="red", linestyle="--", linewidth=1.5, label=f"Observed ARI={obs_ari:.3f}")
#     ax.set_title(f"Fig 4.2.4.4: Permutation null vs observed — {mode} × {index}")
#     ax.set_xlabel("ARI under label permutation (null)"); ax.set_ylabel("Count")
#     ax.legend(); fig.tight_layout()
#     fig.savefig(outdir / f"Fig_4.2.4.4_perm-hist_{mode}_{index}.png", bbox_inches="tight")
#     fig.savefig(outdir / f"Fig_4.2.4.4_perm-hist_{mode}_{index}.pdf", bbox_inches="tight")
#     plt.close(fig)

# def plot_kfixed_bars(df: pd.DataFrame, mode: str, k_filter: str, outdir: Path):
#     if df.empty: return
#     set_font()
#     df = df.sort_values("index")
#     fig, ax = plt.subplots(figsize=(9,4), dpi=DPI)
#     x = np.arange(len(df))
#     colors = ["#4E79A7" if p<=0.05 else "#9FA8B3" for p in df["p_perm"]]
#     ax.bar(x, df["ARI"], color=colors, alpha=0.95)
#     for xi, r in enumerate(df.itertuples()):
#         ax.text(xi, r.ARI+0.01, f"{r.ARI:.2f}\np={r.p_perm:.3f}", ha="center", va="bottom", fontsize=8)
#     ax.set_xticks(x); ax.set_xticklabels(df["index"], rotation=35, ha="right")
#     ax.set_ylim(0,1.05); ax.set_ylabel("ARI (OH dominant vs FP dominant)")
#     ax.set_title(f"Fig 4.2.4.4: ARI by index — {mode} ({'kOH=kFP=2' if k_filter=='k2_only' else 'k≠2含む'})")
#     ax.grid(axis="y", linestyle="--", linewidth=0.4, alpha=0.6)
#     fname_core = f"Fig_4.2.4.4_{k_filter}_ARIs_by-index_{mode}"
#     fig.tight_layout()
#     fig.savefig(outdir / f"{fname_core}.png", bbox_inches="tight")
#     fig.savefig(outdir / f"{fname_core}.pdf", bbox_inches="tight")
#     plt.close(fig)

# def build_validation_kfixed():
#     set_font()
#     Xoh0 = load_features_bin(FEATURES_OH)
#     Xfp0 = load_features_bin(FEATURES_FP)
#     common_samples = Xoh0.index.intersection(Xfp0.index)
#     if len(common_samples) == 0:
#         print("[VALID] Skip: no common samples.")
#         return
#     Xoh0 = Xoh0.loc[common_samples]; Xfp0 = Xfp0.loc[common_samples]

#     oh_latest = latest_by_mode_index(FIGS_OH, "OH")
#     fp_latest = latest_by_mode_index(FIGS_FP, "FP")
#     keys = [(m,i) for m in DIMS for i in INDICES if (m,i) in oh_latest and (m,i) in fp_latest]
#     if not keys:
#         print("[VALID] Skip: no common (mode,index)."); return

#     rows_cond = []

#     for (mode, index) in keys:
#         oh_assign = read_var_cluster(oh_latest[(mode,index)])
#         fp_assign = read_var_cluster(fp_latest[(mode,index)])

#         Xoh = Xoh0[[c for c in Xoh0.columns if c in oh_assign.index]]
#         Xfp = Xfp0[[c for c in Xfp0.columns if c in fp_assign.index]]
#         lab_oh = derive_sample_labels(Xoh, oh_assign, min_votes=1)
#         lab_fp = derive_sample_labels(Xfp, fp_assign, min_votes=1)

#         mask = lab_oh.notna() & lab_fp.notna()
#         if mask.sum() < 5:  # 少数例回避
#             continue
#         yA = lab_oh[mask].astype(int).values
#         yB = lab_fp[mask].astype(int).values

#         # 観測
#         ari_obs = float(adjusted_rand_score(yA, yB))
#         k_oh = int(pd.Series(oh_assign.dropna().astype(str).astype(int)).nunique())
#         k_fp = int(pd.Series(fp_assign.dropna().astype(str).astype(int)).nunique())
#         n_eff = int(mask.sum())

#         # 置換検定・ブートCI
#         ari_obs2, null_vals, p_perm = permutation_pvalue(yA, yB, n_perm=N_PERM)
#         (ci_lo, ci_hi), boot_vals = bootstrap_ci(yA, yB, n_boot=N_BOOT, alpha=0.05)

#         # 保存
#         rows_cond.append({
#             "mode": mode, "index": index, "n_samples": n_eff, "k_OH": k_oh, "k_FP": k_fp,
#             "ARI": ari_obs, "p_perm": p_perm, "boot_CI_lo": ci_lo, "boot_CI_hi": ci_hi
#         })

#         # 図：置換ヒスト
#         plot_perm_hist(null_vals, ari_obs, mode, index, OUT_VALID)

#     # 条件別テーブル
#     df_cond = pd.DataFrame(rows_cond).sort_values(["mode","index"])
#     out_csv_cond = OUT_VALID / "Table_4.2.4.4_validation_per-condition.csv"
#     df_cond.to_csv(out_csv_cond, index=False, encoding="utf-8-sig")
#     print(f"[SAVE] {out_csv_cond}")

#     # k固定（k=2 と それ以外）集計と図
#     def summarize_block(df):
#         if df.empty:
#             return pd.DataFrame(columns=["mode","index","n_conditions","mean_ARI","median_ARI","sig_rate"])
#         g = (df.groupby(["mode","index"], as_index=False)
#                .agg(n_conditions=("ARI","size"),
#                     mean_ARI=("ARI","mean"),
#                     median_ARI=("ARI","median"),
#                     sig_rate=("p_perm", lambda s: float((s<=0.05).mean()))))
#         return g

#     df_k2     = df_cond[(df_cond["k_OH"]==2) & (df_cond["k_FP"]==2)].copy()
#     df_kNot2  = df_cond[~((df_cond["k_OH"]==2) & (df_cond["k_FP"]==2))].copy()
#     sum_k2    = summarize_block(df_k2);    sum_k2["k_stratum"]    = "kOH=kFP=2"
#     sum_kNot2 = summarize_block(df_kNot2); sum_kNot2["k_stratum"] = "kOH/kFP≠2含む"
#     df_kfix   = pd.concat([sum_k2, sum_kNot2], ignore_index=True)
#     out_csv_kfix = OUT_VALID / "Table_4.2.4.4_validation_k-fixed_summary.csv"
#     df_kfix.to_csv(out_csv_kfix, index=False, encoding="utf-8-sig")
#     print(f"[SAVE] {out_csv_kfix}")

#     for mode in DIMS:
#         plot_kfixed_bars(df_k2[df_k2["mode"]==mode][["index","ARI","p_perm"]], mode, "k2_only", OUT_VALID)
#         plot_kfixed_bars(df_kNot2[df_kNot2["mode"]==mode][["index","ARI","p_perm"]], mode, "kNot2", OUT_VALID)

# # ========= MAIN =========
# def main(run_forward=True, run_reverse=True, run_contrast=True, run_bidir=True, run_validation=True):
#     set_font()
#     if run_contrast:
#         print("=== [1/5] OH↔FP (new vs signless) contrast ===")
#         build_ohfp_contrast()
#     if run_forward:
#         print("=== [2/5] OH→FP forward direction ===")
#         build_forward_direction()
#     if run_reverse:
#         print("=== [3/5] FP→OH reverse direction ===")
#         build_reverse_direction()
#     if run_bidir:
#         print("=== [4/5] Bidirectional summary ===")
#         build_bidirectional_summary()
#     if run_validation:
#         print("=== [5/5] k-fixed validation (permutation & bootstrap) ===")
#         build_validation_kfixed()
#     print("\n✅ Done. Outputs under:", OUT_OHFP_ROOT)

# if __name__ == "__main__":
#     main()

In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
diagnose_4_2_4_4_empty_bars.py
--------------------------------
4.2.4.4 の棒グラフが「ほとんど空」になる原因を条件ごとに切り分ける診断ツール。

診断すること：
 1) main 側（figs_OH / figs_FP）の ClusterAssign_* を (mode,index) ごとに検出
 2) signless 側（STEP3.2_signlessCorr_MDS_YYYYmmdd/{OH,FP}/cluster_labels_*.csv）の linear_* 有無を確認
 3) (mode,index) ごとの kA, kB, 共通ID数(n)、除外理由タグを付与
 4) 「k≤30 フィルタ」あり/なしの両ケースで、本文棒グラフに残る条件がどれかを一覧化

出力:
  ROOT/paper_4.2.4.4_OHFP/diagnostics/
    - diag_summary_main_text.csv   ← k≤30 条件での採否と原因
    - diag_summary_all_k.csv       ← all-k 条件での採否と原因
    - diag_missing_by_side.csv     ← main / signless どちらが欠けているか
    - diag_counts.txt              ← コンソール要約をテキスト保存

使い方:
  python diagnose_4_2_4_4_empty_bars.py
  python diagnose_4_2_4_4_empty_bars.py --all-k   # kフィルタ無効で再診断

"""

from pathlib import Path
import os, re, glob
import argparse
import pandas as pd
import numpy as np
from sklearn.metrics import adjusted_rand_score

# ========= 設定 =========
ROOT = Path("/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No")
FIGS_OH = ROOT / "figs_OH"
FIGS_FP = ROOT / "figs_FP"

SIGNLESS_BASE = ROOT / "STEP3.2_signlessCorr_MDS_20250904"  # 環境に合わせて調整
SIGNLESS_DIR = {"OH": SIGNLESS_BASE / "OH", "FP": SIGNLESS_BASE / "FP"}

OUTDIR = ROOT / "paper_4.2.4.4_OHFP" / "diagnostics"
OUTDIR.mkdir(parents=True, exist_ok=True)

DIMS    = ["top3","cumeig"]
INDICES = ["silhouette","dunn","gap","ch","db","ptbiserial"]
K_MAX_KEEP = 30

FN_RE = re.compile(
    r"^ClusterAssign_(?P<mode>top3|cumeig)_(?P<index>silhouette|dunn|gap|ch|db|ptbiserial)_(?P<ds>OH|FP)_(?P<date>\d{8})_(?P<hm>\d{4})\.csv$"
)

# ========= ユーティリティ =========
def read_csv_quiet(path: Path) -> pd.DataFrame | None:
    if not path or not path.exists(): return None
    for enc in ("utf-8","utf-8-sig","cp932"):
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            continue
    return None

def latest_by_mode_index(fig_dir: Path, ds: str):
    """(mode,index)->最新ファイルPath"""
    latest = {}
    for p in fig_dir.glob("ClusterAssign_*.csv"):
        m = FN_RE.match(p.name)
        if not m or m["ds"] != ds:
            continue
        key = (m["mode"], m["index"])
        ts  = f"{m['date']}_{m['hm']}"
        if key not in latest or ts > latest[key][0]:
            latest[key] = (ts, p)
    return {k: v[1] for k,v in latest.items()}

def read_var_cluster(path: Path) -> pd.Series | None:
    df = read_csv_quiet(path)
    if df is None or df.empty: return None
    cols = {c.lower(): c for c in df.columns}
    vcol = cols.get("variable", df.columns[0])
    ccol = cols.get("cluster",  df.columns[-1])
    s = pd.Series(df[ccol].values, index=df[vcol].astype(str).values, name="cluster")
    try: s = s.astype(int)
    except: pass
    return s

def load_signless_labels(signless_dir: Path) -> pd.DataFrame | None:
    """cluster_labels_* の linear_* 列だけ保持"""
    cands = sorted(signless_dir.glob("cluster_labels_*.csv"), key=os.path.getmtime, reverse=True)
    if not cands: return None
    df = read_csv_quiet(cands[0])
    if df is None or "id" not in df.columns: return None
    keep = [c for c in df.columns if str(c).startswith("linear_")]
    if not keep: return None
    out = df[["id"]+keep].copy()
    out.columns = ["id"] + [c.replace("linear_","",1) for c in keep]
    return out

def k_from_series(s: pd.Series) -> int:
    if s is None or s.empty: return np.nan
    return int(pd.Series(s).nunique(dropna=True))

def diag_once(k_filter_on: bool = True) -> pd.DataFrame:
    """k≤30フィルタの有無を切り替えて診断"""
    oh_latest = latest_by_mode_index(FIGS_OH, "OH")
    fp_latest = latest_by_mode_index(FIGS_FP, "FP")
    keys = sorted(list(set(oh_latest.keys()).intersection(fp_latest.keys())))
    rows = []

    # signless availability
    S_OH = load_signless_labels(SIGNLESS_DIR["OH"])
    S_FP = load_signless_labels(SIGNLESS_DIR["FP"])
    signless_avail = set()
    if S_OH is not None:
        signless_avail.update([c for c in S_OH.columns if c != "id"])
    if S_FP is not None:
        signless_avail.intersection_update([c for c in S_FP.columns if c != "id"]) if signless_avail else signless_avail.update([c for c in S_FP.columns if c != "id"])

    for mode, index in keys:
        # main: OH/FP それぞれの ClusterAssign を読む
        sOH = read_var_cluster(oh_latest[(mode,index)])
        sFP = read_var_cluster(fp_latest[(mode,index)])

        # k（クラスタ数）
        kOH = k_from_series(sOH)
        kFP = k_from_series(sFP)

        # signless 対応の有無（同じ condition 名が linear_* に存在するか）
        cond = f"{mode}_{index}"
        has_signless = cond in signless_avail

        # 本来の棒グラフ作成では、OH と FP で「new vs signless（線形）」を突き合わせるため、
        # まず “new(=main)” 同士の共通IDを確認（ここでは Variable 名一致を proxy に使う）
        if sOH is not None and sFP is not None:
            ids_common = sorted(set(sOH.index).intersection(set(sFP.index)))
            n_common_vars = len(ids_common)
        else:
            n_common_vars = 0

        # kフィルタに引っかかるか？
        filtered_by_k = False
        if k_filter_on and ( (not np.isnan(kOH) and kOH > K_MAX_KEEP) or (not np.isnan(kFP) and kFP > K_MAX_KEEP) ):
            filtered_by_k = True

        # タグ付け（最も強い原因を上に）
        reasons = []
        if sOH is None or sFP is None:
            # どちらが無いか詳細を記録
            if sOH is None: reasons.append("NO_MAIN_FILE_OH")
            if sFP is None: reasons.append("NO_MAIN_FILE_FP")
        if not has_signless:
            reasons.append("NO_SIGNLESS_FILE")
        if filtered_by_k:
            reasons.append("K_FILTERED_OUT")
        if n_common_vars < 2:
            reasons.append("LOW_MATCH_N")

        # 「本文に残る（OK_INCLUDED）」か？
        ok_included = False
        if (sOH is not None) and (sFP is not None) and has_signless and (not filtered_by_k) and (n_common_vars >= 2):
            ok_included = True
        if ok_included and not reasons:
            reasons = ["OK_INCLUDED"]

        rows.append({
            "mode": mode,
            "index": index,
            "kOH": kOH,
            "kFP": kFP,
            "has_main_OH": sOH is not None,
            "has_main_FP": sFP is not None,
            "has_signless_same_condition": has_signless,
            "n_common_variables(OH∩FP)": n_common_vars,
            "filtered_by_k<=30": filtered_by_k if k_filter_on else False,
            "status": "|".join(reasons) if reasons else "OK_INCLUDED"
        })

    return pd.DataFrame(rows).sort_values(["mode","index"])

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--all-k", action="store_true", help="kフィルタを無効化（= all-k と同様の条件で診断）")
    args = ap.parse_args()

    # k≤30（本文条件）で診断
    df_main = diag_once(k_filter_on= not args.all_k)
    out1 = OUTDIR / ("diag_summary_main_text.csv" if not args.all_k else "diag_summary_all_k.csv")
    df_main.to_csv(out1, index=False, encoding="utf-8-sig")
    print(f"[SAVE] {out1}")

    # どちらが欠けているのか俯瞰
    miss_rows = []
    for ds, figs in [("OH", FIGS_OH), ("FP", FIGS_FP)]:
        latest = latest_by_mode_index(figs, ds)
        have = set(latest.keys())
        miss = [(m, i) for m in DIMS for i in INDICES if (m, i) not in have]
        for m, i in miss:
            miss_rows.append({"dataset": ds, "mode": m, "index": i, "missing_main_file": True})

    if len(miss_rows):
        miss_df = pd.DataFrame(miss_rows).sort_values(["dataset", "mode", "index"])
    else:
        # 空でもCSVを出す（列だけ維持）
        miss_df = pd.DataFrame(columns=["dataset", "mode", "index", "missing_main_file"])

    out2 = OUTDIR / "diag_missing_by_side.csv"
    miss_df.to_csv(out2, index=False, encoding="utf-8-sig")
    print(f"[SAVE] {out2}")

    # 簡易テキストまとめ（df_mainが空でも安全に）
    if len(df_main):
        n_ok = (df_main["status"] == "OK_INCLUDED").sum()
        n_k  = df_main["status"].str.contains("K_FILTERED_OUT").sum()
        n_no_signless = df_main["status"].str.contains("NO_SIGNLESS_FILE").sum()
        n_low = df_main["status"].str.contains("LOW_MATCH_N").sum()
        n_main_oh = df_main["status"].str.contains("NO_MAIN_FILE_OH").sum()
        n_main_fp = df_main["status"].str.contains("NO_MAIN_FILE_FP").sum()
    else:
        n_ok = n_k = n_no_signless = n_low = n_main_oh = n_main_fp = 0


    summary_text = f"""
[診断まとめ] ({'all-k' if args.all_k else 'k<=30'}) 条件
- OK_INCLUDED: {n_ok}
- K_FILTERED_OUT: {n_k}
- NO_SIGNLESS_FILE: {n_no_signless}
- LOW_MATCH_N: {n_low}
- NO_MAIN_FILE_OH: {n_main_oh}
- NO_MAIN_FILE_FP: {n_main_fp}

詳細は {out1.name} を参照。どの (mode,index) がどの理由で棒から落ちたか一目で分かります。
"""
    print(summary_text)
    (OUTDIR / "diag_counts.txt").write_text(summary_text.strip(), encoding="utf-8")

if __name__ == "__main__":
    import sys
    sys.argv = ["diagnose_4_2_4_4_empty_bars.py"]  # Jupyter引数を無効化
    main()


[SAVE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP/diagnostics/diag_summary_main_text.csv
[SAVE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP/diagnostics/diag_missing_by_side.csv

[診断まとめ] (k<=30) 条件
- OK_INCLUDED: 4
- K_FILTERED_OUT: 8
- NO_SIGNLESS_FILE: 0
- LOW_MATCH_N: 0
- NO_MAIN_FILE_OH: 0
- NO_MAIN_FILE_FP: 0

詳細は diag_summary_main_text.csv を参照。どの (mode,index) がどの理由で棒から落ちたか一目で分かります。

